# LeetCode 352 — Data Stream as Disjoint Intervals bằng Python (Hard)

Notebook này giảng nhập môn một bài **thiết kế class** (design): ta phải viết một class "nhớ" các số đã nhận và lúc nào cũng tóm tắt được chúng thành các **khoảng rời nhau**.

```python
class SummaryRanges:

    def __init__(self):


    def addNum(self, value: int) -> None:


    def getIntervals(self) -> List[List[int]]:



# Your SummaryRanges object will be instantiated and called as such:
# obj = SummaryRanges()
# obj.addNum(value)
# param_2 = obj.getIntervals()
```

Mục tiêu của bài học:

1. Hiểu chắc **khoảng số nguyên** (interval) `[start, end]`: chứa những số nào, khi nào hai khoảng **chồng nhau** hoặc **liền kề** và phải gộp thành một.
2. Hiểu bài **thiết kế class**: object phải nhớ trạng thái giữa các lần gọi; đọc được input dạng chuỗi lệnh `["SummaryRanges","addNum",...]`, `[[],[1],...]` và tự viết hàm `run_commands` để test giống LeetCode.
3. Đi từ cách người mới tự nghĩ ra (lưu `set` các số, mỗi lần hỏi thì sắp xếp rồi gộp) → **duy trì sẵn danh sách khoảng đã sắp xếp**, dùng `bisect` tìm chỗ rồi gộp trái/phải (cách chính) → **hai dict hai đầu** gộp `O(1)` → ý tưởng **cây cân bằng / SortedDict** cho câu follow-up.
4. Học **binary search / module `bisect`** từ con số 0 và đặc biệt là mẹo `bisect_right(starts, value) - 1` để tìm "khoảng bên trái".
5. Phân loại được **5 trường hợp** khi thêm một số: nằm trong, nối trái, nối phải, nối cả hai, đứng riêng — và giải thích vì sao **chỉ có tối đa 2 khoảng** bị ảnh hưởng.
6. Phân tích độ phức tạp **theo từng thao tác** (`addNum` riêng, `getIntervals` riêng) và chọn cách phù hợp với constraints.
7. Tự kiểm chứng bằng test chéo ngẫu nhiên có `random.seed`, tránh các lỗi kinh điển (chỉ số `-1` của Python, `bisect_left` thay vì `bisect_right`, quên gộp hai phía...).

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một dòng dữ liệu (data stream) gồm các số nguyên **không âm** `a1, a2, ..., an`, hãy tóm tắt các số đã thấy tới lúc này thành một danh sách các **khoảng rời nhau** (disjoint intervals).
>
> Cài đặt class `SummaryRanges`:
>
> - `SummaryRanges()` khởi tạo object với một stream rỗng.
> - `void addNum(int value)` thêm số nguyên `value` vào stream.
> - `int[][] getIntervals()` trả về bản tóm tắt các số nguyên trong stream **hiện tại** dưới dạng danh sách các khoảng rời nhau `[start_i, end_i]`. Kết quả phải được **sắp xếp theo `start_i`**.

**Example 1**

```text
Input
["SummaryRanges", "addNum", "getIntervals", "addNum", "getIntervals", "addNum", "getIntervals", "addNum", "getIntervals", "addNum", "getIntervals"]
[[], [1], [], [3], [], [7], [], [2], [], [6], []]
Output
[null, null, [[1, 1]], null, [[1, 1], [3, 3]], null, [[1, 1], [3, 3], [7, 7]], null, [[1, 3], [7, 7]], null, [[1, 3], [6, 7]]]

Explanation
SummaryRanges summaryRanges = new SummaryRanges();
summaryRanges.addNum(1);      // arr = [1]
summaryRanges.getIntervals(); // return [[1, 1]]
summaryRanges.addNum(3);      // arr = [1, 3]
summaryRanges.getIntervals(); // return [[1, 1], [3, 3]]
summaryRanges.addNum(7);      // arr = [1, 3, 7]
summaryRanges.getIntervals(); // return [[1, 1], [3, 3], [7, 7]]
summaryRanges.addNum(2);      // arr = [1, 2, 3, 7]
summaryRanges.getIntervals(); // return [[1, 3], [7, 7]]
summaryRanges.addNum(6);      // arr = [1, 2, 3, 6, 7]
summaryRanges.getIntervals(); // return [[1, 3], [6, 7]]
```

**Constraints**

- `0 <= value <= 10^4`
- Có tối đa `3 * 10^4` lời gọi `addNum` và `getIntervals` (tính chung).
- Có tối đa `10^2` lời gọi `getIntervals`.

**Follow up:** Nếu có **rất nhiều lần gộp** (merge) và **số khoảng rời nhau nhỏ** so với kích thước của data stream thì sao?

### Ký hiệu dùng suốt notebook

- `n` = số **giá trị phân biệt** đã được thêm tính tới thời điểm đang xét (số trùng chỉ tính 1 lần).
- `k` = số **khoảng** đang có (luôn `k <= n`; nhiều số liền nhau gộp lại thì `k` nhỏ hơn `n` rất nhiều).
- `q` = tổng số lời gọi (`q <= 3·10^4`), `g` = số lời gọi `getIntervals` (`g <= 100`).
- `V = 10^4 + 1` = số giá trị có thể có (`0..10^4`).

### 1.1 Đọc input / output của bài "thiết kế class"

Đây **không** phải dạng "cho mảng, trả kết quả". LeetCode mô tả một **kịch bản gọi hàm** bằng 2 list song song: list thứ nhất là **tên lệnh**, list thứ hai là **tham số** của lệnh tương ứng.

| Vị trí | Tên lệnh | Tham số | Nghĩa là | Các số đã có | Output |
|---:|---|---|---|---|---|
| 0 | `"SummaryRanges"` | `[]` | `obj = SummaryRanges()` | `{}` | `null` |
| 1 | `"addNum"` | `[1]` | `obj.addNum(1)` | `{1}` | `null` (hàm `void`) |
| 2 | `"getIntervals"` | `[]` | `obj.getIntervals()` | `{1}` | `[[1, 1]]` |
| 3 | `"addNum"` | `[3]` | `obj.addNum(3)` | `{1, 3}` | `null` |
| 4 | `"getIntervals"` | `[]` | | `{1, 3}` | `[[1, 1], [3, 3]]` |
| 5 | `"addNum"` | `[7]` | | `{1, 3, 7}` | `null` |
| 6 | `"getIntervals"` | `[]` | | `{1, 3, 7}` | `[[1, 1], [3, 3], [7, 7]]` |
| 7 | `"addNum"` | `[2]` | | `{1, 2, 3, 7}` | `null` |
| 8 | `"getIntervals"` | `[]` | | `{1, 2, 3, 7}` | `[[1, 3], [7, 7]]` |
| 9 | `"addNum"` | `[6]` | | `{1, 2, 3, 6, 7}` | `null` |
| 10 | `"getIntervals"` | `[]` | | `{1, 2, 3, 6, 7}` | `[[1, 3], [6, 7]]` |

Quan sát quan trọng nhất của ví dụ:

- Một số đứng một mình (như `7` lúc đầu) vẫn là một khoảng, viết `[7, 7]` (start = end).
- Khi thêm `2`, ba số `1, 2, 3` trở thành **liên tiếp nhau**: hai khoảng `[1, 1]` và `[3, 3]` cùng với số `2` bị "hàn" lại thành **một** khoảng `[1, 3]`. Một lần `addNum` có thể làm **giảm** số khoảng!
- Khi thêm `6`, khoảng `[7, 7]` được **kéo dài sang trái** thành `[6, 7]`.
- `null` là cách Java / JSON viết "không có giá trị"; trong Python tương ứng là `None`.

### 1.2 Vì sao bài này là Hard?

Nếu chỉ hỏi bản tóm tắt **một lần** ở cuối thì rất dễ: sắp xếp các số rồi đi từ trái sang phải, gom các đoạn số liên tiếp. Cái khó là:

1. **Dữ liệu chảy vào từng số một**, còn `getIntervals` có thể bị gọi xen kẽ bất cứ lúc nào.
2. Một số mới có thể: không làm gì cả (đã có rồi), kéo dài một khoảng, **hàn hai khoảng làm một**, hoặc tạo khoảng mới. Phải xử lý đúng **mọi** trường hợp, kể cả số trùng.
3. Câu **follow-up** hỏi về hiệu năng khi gộp rất nhiều: lúc đó stream có thể có hàng nghìn số nhưng chỉ vài khoảng. Ta muốn bộ nhớ và thời gian phụ thuộc vào **số khoảng `k`**, không phụ thuộc vào **số giá trị `n`**.

Lưu ý thêm về constraints: `getIntervals` bị gọi **ít** (tối đa 100 lần) còn `addNum` có thể tới gần `3·10^4` lần. Đây là gợi ý ngầm: `addNum` nên nhẹ; `getIntervals` có thể "nặng" hơn một chút mà vẫn ổn. Cả 4 cách trong notebook đều qua được giới hạn của LeetCode — điều đáng học là **mỗi cách trả giá ở thao tác nào** và vì sao.

In [1]:
# Nhìn dữ liệu "thô" sau mỗi lần addNum của Example 1 (chưa cần thuật toán gì)
stream = [1, 3, 7, 2, 6]
seen = []
for v in stream:
    seen.append(v)
    print(f"thêm {v} -> các số đã có (sắp xếp): {sorted(seen)}")

thêm 1 -> các số đã có (sắp xếp): [1]
thêm 3 -> các số đã có (sắp xếp): [1, 3]
thêm 7 -> các số đã có (sắp xếp): [1, 3, 7]
thêm 2 -> các số đã có (sắp xếp): [1, 2, 3, 7]
thêm 6 -> các số đã có (sắp xếp): [1, 2, 3, 6, 7]


## 2. Nền tảng

### 2.1 Khoảng số nguyên `[start, end]`

Trong bài này một **khoảng** (interval) là một cặp `[s, e]` với `s <= e`, đại diện cho **mọi số nguyên** từ `s` tới `e`, **tính cả hai đầu** (đóng hai đầu):

- `[1, 3]` = `{1, 2, 3}` — có `e - s + 1 = 3` số.
- `[7, 7]` = `{7}` — chỉ 1 số.
- Kiểm tra `x` có nằm trong `[s, e]` không: `s <= x <= e` (Python cho phép viết so sánh "nối chuỗi" như vậy).

Hai khoảng `A = [a_s, a_e]` đứng trước `B = [b_s, b_e]` (tức `a_s < b_s`) có thể ở 3 tình huống:

| Tình huống | Điều kiện | Ví dụ | Có gộp không? |
|---|---|---|---|
| Chồng nhau (overlap) | `b_s <= a_e` | `[1, 5]` và `[3, 8]` | **Có** → `[1, 8]` |
| Liền kề (adjacent) — "chạm" nhau | `b_s == a_e + 1` | `[1, 3]` và `[4, 6]` | **Có** → `[1, 6]` (vì `{1..3} ∪ {4..6} = {1..6}`, không có lỗ hổng) |
| Có khe hở (gap) | `b_s > a_e + 1` | `[1, 3]` và `[5, 6]` | **Không** — số `4` bị thiếu |

Điểm dễ quên nhất: với **số nguyên**, hai khoảng **liền kề** cũng phải gộp. `[1, 3]` và `[4, 6]` không chung số nào, nhưng đề muốn bản tóm tắt **gọn nhất**, nên chúng là một khoảng `[1, 6]`. Vì vậy trong bài này "rời nhau" (disjoint) thực ra nghĩa mạnh hơn: giữa hai khoảng kề nhau trong kết quả luôn có **ít nhất một số bị thiếu**.

In [2]:
def size(iv):
    s, e = iv                    # "rải" cặp [s, e] thành 2 biến
    return e - s + 1


def contains(iv, x):
    s, e = iv
    return s <= x <= e           # so sánh nối chuỗi: s <= x và x <= e


def relation(a, b):
    # a đứng trước b (a[0] < b[0]); trả về quan hệ giữa hai khoảng
    if b[0] <= a[1]:
        return "chồng nhau -> gộp"
    if b[0] == a[1] + 1:
        return "liền kề -> gộp"
    return "có khe hở -> để riêng"


print(size([1, 3]), size([7, 7]))                 # 3 1
print(contains([1, 3], 2), contains([1, 3], 4))   # True False
print(relation([1, 5], [3, 8]))
print(relation([1, 3], [4, 6]))
print(relation([1, 3], [5, 6]))

3 1
True False
chồng nhau -> gộp
liền kề -> gộp
có khe hở -> để riêng


### 2.2 `set` — tập hợp các số đã thấy, tự loại trùng

Stream có thể chứa **số trùng** (ví dụ thêm `2` hai lần). Bản tóm tắt chỉ quan tâm "số này **có** xuất hiện hay không", nên công cụ tự nhiên nhất là `set` (tập hợp):

- Tạo: `seen = set()` (chú ý: `{}` là **dict rỗng**, không phải set rỗng).
- Thêm: `seen.add(x)` — thêm số đã có thì **không** có gì thay đổi.
- Kiểm tra: `x in seen` — trung bình `O(1)` nhờ băm (hash).
- Set **không có thứ tự**; muốn duyệt theo thứ tự tăng dần phải `sorted(seen)` → trả về một **list mới** đã sắp xếp, tốn `O(n log n)`.

In [3]:
seen = set()
for x in [5, 1, 5, 3, 1]:
    seen.add(x)              # thêm trùng không sao
print(len(seen))             # 3: chỉ còn 5, 1, 3
print(3 in seen, 4 in seen)  # True False
print(sorted(seen))          # [1, 3, 5] -> list mới, tăng dần
print(type({}), type(set()))

3
True False
[1, 3, 5]
<class 'dict'> <class 'set'>


### 2.3 Class và trạng thái: object "nhớ" dữ liệu giữa các lần gọi

Một **class** là "bản thiết kế"; mỗi lần gọi `SummaryRanges()` ta tạo ra một **object** (đối tượng) mới theo bản thiết kế đó.

- `__init__(self)` chạy **đúng 1 lần** khi tạo object. Đây là chỗ tạo "ngăn chứa" dữ liệu.
- `self` là chính object đang được dùng. Viết `self.seen = set()` nghĩa là "object này có một ngăn tên `seen`".
- Các method khác (`addNum`, `getIntervals`) đọc / sửa các ngăn `self.xxx`. Vì ngăn nằm trên object nên giá trị **còn nguyên** giữa các lần gọi. Biến cục bộ (không có `self.`) thì biến mất khi method chạy xong.

Ví dụ mini: class `DistinctCounter` đếm xem stream có bao nhiêu **giá trị khác nhau**. Nó chỉ cần nhớ một `set`.

In [4]:
class DistinctCounter:
    def __init__(self):            # chạy 1 lần khi tạo object
        self.seen = set()          # ngăn "seen" của riêng object này

    def add(self, x):
        self.seen.add(x)

    def count(self):
        return len(self.seen)


a = DistinctCounter()
a.add(4)
a.add(4)
a.add(9)
print("a.count() =", a.count())    # 2

b = DistinctCounter()              # object MỚI -> trạng thái mới
b.add(1)
print("b.count() =", b.count(), "| a vẫn là", a.count())

a.count() = 2
b.count() = 1 | a vẫn là 2


### 2.4 Tự viết hàm chạy chuỗi lệnh kiểu LeetCode

Để test giống hệt LeetCode, ta viết hàm `run_commands(cls, ops, args)`:

1. Tạo object: `obj = cls(*args[0])`.
2. Với mỗi cặp (`op`, `a`) còn lại: lấy method có tên `op` bằng `getattr(obj, op)` rồi gọi với tham số `*a`.
3. Gom kết quả từng lời gọi vào list `outputs` (lời gọi khởi tạo cho `None`, tương ứng `null`).

Hai công cụ Python mới:

- `getattr(obj, "add")` lấy method của object theo **tên dạng chuỗi**: `getattr(c, "add")(10)` giống hệt `c.add(10)`. Ta cần nó vì `ops` chứa tên lệnh dạng chuỗi như `"addNum"`.
- `*a` khi gọi hàm: "rải" (unpack) list thành các đối số riêng lẻ. `f(*[1])` giống `f(1)`, còn `f(*[])` giống `f()`. Nhờ vậy cùng một dòng code gọi được cả `addNum(1)` lẫn `getIntervals()`.

Thêm 2 hàm phụ:

- `show(outputs)` in kết quả giống LeetCode (`None` → `null`).
- `build_commands(tokens)` giúp viết test gọn: số nguyên → `addNum(số)`, chuỗi `"?"` → `getIntervals()`. Ví dụ `[1, "?", 3, "?", 7, "?", 2, "?", 6, "?"]` chính là Example 1.

Ta truyền **class thật** vào tham số `cls`, nhờ vậy chạy được nhiều cách cài đặt khác nhau trên **cùng một** chuỗi lệnh — rất tiện để test chéo ở cuối.

In [5]:
c = DistinctCounter()
getattr(c, "add")(10)               # giống c.add(10)
getattr(c, "add")(*[20])            # *[20] rải list thành 1 đối số
print(getattr(c, "count")(*[]))     # *[] -> 0 đối số: c.count() -> 2


def run_commands(cls, ops, args):
    # Chạy chuỗi lệnh kiểu LeetCode: ops[0] là lệnh tạo object, args[0] là tham số của __init__
    obj = cls(*args[0])
    outputs = [None]                        # lời gọi khởi tạo -> null
    for op, a in zip(ops[1:], args[1:]):
        method = getattr(obj, op)           # lấy method theo tên, ví dụ "addNum"
        outputs.append(method(*a))          # gọi với tham số rải từ list a
    return outputs


def show(outputs):
    # In giống LeetCode: None -> null
    return "[" + ", ".join("null" if x is None else str(x) for x in outputs) + "]"


def build_commands(tokens, class_name="SummaryRanges"):
    # Số nguyên -> addNum(số); "?" -> getIntervals()
    ops, args = [class_name], [[]]
    for t in tokens:
        if t == "?":
            ops.append("getIntervals")
            args.append([])
        else:
            ops.append("addNum")
            args.append([t])
    return ops, args


EX1_OPS = ["SummaryRanges", "addNum", "getIntervals", "addNum", "getIntervals", "addNum",
           "getIntervals", "addNum", "getIntervals", "addNum", "getIntervals"]
EX1_ARGS = [[], [1], [], [3], [], [7], [], [2], [], [6], []]
EX1_OUT = [None, None, [[1, 1]], None, [[1, 1], [3, 3]], None, [[1, 1], [3, 3], [7, 7]],
           None, [[1, 3], [7, 7]], None, [[1, 3], [6, 7]]]

ops1, args1 = build_commands([1, "?", 3, "?", 7, "?", 2, "?", 6, "?"])
assert ops1 == EX1_OPS and args1 == EX1_ARGS
print("build_commands tạo đúng input Example 1")
print(show(run_commands(DistinctCounter, ["DistinctCounter", "add", "add", "count"], [[], [3], [3], []])))

2
build_commands tạo đúng input Example 1
[null, null, null, 1]


## 3. Cách 1 — Lưu `set` các số, mỗi lần hỏi thì sắp xếp rồi gộp

### 3.1 Ý tưởng người mới tự nghĩ ra

"Cứ lưu hết các số lại. Khi bị hỏi thì sắp xếp và gom các đoạn liên tiếp." Đây là cách tự nhiên nhất, và nó **đúng**.

Việc gom các đoạn liên tiếp từ một list **đã sắp xếp, không trùng** làm như sau: đi từ trái sang phải, giữ khoảng hiện tại `[s, e]`:

- Nếu số tiếp theo `x == e + 1` → nó nối liền khoảng hiện tại → kéo dài: `e = x`.
- Ngược lại (có khe hở) → "đóng" khoảng `[s, e]` lại, ghi vào kết quả, rồi mở khoảng mới `[x, x]`.
- Hết list → nhớ ghi khoảng cuối cùng (lỗi rất hay quên!).

Tính tay với `[1, 2, 3, 6, 7]`:

| `x` | So với `e + 1` | Hành động | Khoảng đang mở | Kết quả đã đóng |
|---:|---|---|---|---|
| 1 | (số đầu) | mở `[1, 1]` | `[1, 1]` | `[]` |
| 2 | `2 == 1 + 1` | kéo dài | `[1, 2]` | `[]` |
| 3 | `3 == 2 + 1` | kéo dài | `[1, 3]` | `[]` |
| 6 | `6 != 3 + 1` | đóng `[1, 3]`, mở `[6, 6]` | `[6, 6]` | `[[1, 3]]` |
| 7 | `7 == 6 + 1` | kéo dài | `[6, 7]` | `[[1, 3]]` |
| hết | | đóng `[6, 7]` | | `[[1, 3], [6, 7]]` |

In [6]:
def compress_sorted(nums, verbose=False):
    # nums: list ĐÃ SẮP XẾP và KHÔNG TRÙNG -> list các khoảng [s, e]
    result = []
    if not nums:
        return result
    s = e = nums[0]                      # mở khoảng đầu tiên
    for x in nums[1:]:
        if x == e + 1:                   # liền kề -> kéo dài
            e = x
            action = "kéo dài"
        else:                            # có khe hở -> đóng khoảng cũ, mở khoảng mới
            result.append([s, e])
            s = e = x
            action = "đóng + mở mới"
        if verbose:
            print(f"x={x}: {action:14s} đang mở=[{s}, {e}]  đã đóng={result}")
    result.append([s, e])                # ĐỪNG QUÊN khoảng cuối
    return result


print(compress_sorted([1, 2, 3, 6, 7], verbose=True))
print(compress_sorted([4]))
print(compress_sorted([]))
print(compress_sorted([0, 2, 4, 5, 6, 10]))

x=2: kéo dài        đang mở=[1, 2]  đã đóng=[]
x=3: kéo dài        đang mở=[1, 3]  đã đóng=[]
x=6: đóng + mở mới  đang mở=[6, 6]  đã đóng=[[1, 3]]
x=7: kéo dài        đang mở=[6, 7]  đã đóng=[[1, 3]]
[[1, 3], [6, 7]]
[[4, 4]]
[]
[[0, 0], [2, 2], [4, 6], [10, 10]]


### 3.2 Đóng gói thành class

- `__init__`: tạo `self.seen = set()`.
- `addNum(value)`: chỉ `self.seen.add(value)` — cực nhẹ, `O(1)` trung bình; số trùng tự bị bỏ qua.
- `getIntervals()`: `compress_sorted(sorted(self.seen))`.

In [7]:
class SummaryRangesSortSet:
    # Cách 1: lưu set, getIntervals sắp xếp rồi gộp
    def __init__(self):
        self.seen = set()

    def addNum(self, value):
        self.seen.add(value)

    def getIntervals(self):
        return compress_sorted(sorted(self.seen))


out = run_commands(SummaryRangesSortSet, EX1_OPS, EX1_ARGS)
print(show(out))
assert out == EX1_OUT
print("Cách 1 PASS Example 1")

[null, null, [[1, 1]], null, [[1, 1], [3, 3]], null, [[1, 1], [3, 3], [7, 7]], null, [[1, 3], [7, 7]], null, [[1, 3], [6, 7]]]
Cách 1 PASS Example 1


### 3.3 Độ phức tạp cách 1 và một biến thể "mảng đánh dấu"

Với `n` giá trị phân biệt:

| Thao tác | Time | Giải thích |
|---|---|---|
| `addNum` | `O(1)` trung bình | chỉ thêm vào set |
| `getIntervals` | `O(n log n)` | `sorted` tốn `O(n log n)`, gộp thêm `O(n)` |
| Bộ nhớ | `O(n)` | lưu mọi giá trị phân biệt |

Vì `value` chỉ nằm trong `0..10^4`, còn một biến thể nữa người mới cũng hay nghĩ ra: **mảng đánh dấu** (boolean array) `present` dài `V = 10^4 + 1`, `present[v] = True` khi `v` đã xuất hiện. Khi hỏi, quét `v = 0, 1, ..., 10^4` là thấy ngay các đoạn `True` liên tiếp — **không cần sắp xếp** vì chỉ số mảng đã tự có thứ tự.

- `addNum`: `O(1)` (gán 1 ô).
- `getIntervals`: `O(V)` = luôn quét 10001 ô, kể cả khi mới có 1 số.
- Với `g <= 100` lần hỏi: tối đa khoảng `100 · 10^4 = 10^6` bước → vẫn qua LeetCode. Nhưng cách này **gắn chặt** vào giới hạn `10^4`; nếu value lên tới `10^9` thì hỏng.

In [8]:
class SummaryRangesBoolArray:
    # Biến thể cách 1: mảng đánh dấu, dựa vào 0 <= value <= 10^4
    MAXV = 10**4

    def __init__(self):
        self.present = [False] * (self.MAXV + 2)   # thêm 1 ô "lính canh" luôn False ở cuối

    def addNum(self, value):
        self.present[value] = True

    def getIntervals(self):
        result = []
        v = 0
        while v <= self.MAXV:
            if self.present[v]:
                s = v
                while self.present[v + 1]:         # ô lính canh giúp vòng này luôn dừng
                    v += 1
                result.append([s, v])
            v += 1
        return result


out = run_commands(SummaryRangesBoolArray, EX1_OPS, EX1_ARGS)
print(show(out))
assert out == EX1_OUT
print("Biến thể mảng đánh dấu PASS Example 1")

[null, null, [[1, 1]], null, [[1, 1], [3, 3]], null, [[1, 1], [3, 3], [7, 7]], null, [[1, 3], [7, 7]], null, [[1, 3], [6, 7]]]
Biến thể mảng đánh dấu PASS Example 1


### 3.4 Cách 1 vướng ở đâu?

Hãy nhìn lại Example 1: giữa hai lần `getIntervals`, ta chỉ thêm **đúng một số**. Vậy mà cách 1 sắp xếp lại **toàn bộ** dữ liệu và gộp lại **từ đầu**. Kết quả lần hỏi trước bị vứt đi phí phạm.

Ngoài ra, trong tình huống follow-up (stream rất dài nhưng gộp lại chỉ còn vài khoảng), cách 1 vẫn lưu **từng số** (`O(n)` bộ nhớ) và sắp xếp **từng số** (`O(n log n)`), trong khi câu trả lời chỉ có `k` khoảng.

**Câu hỏi then chốt:** khi thêm một số `value`, bức tranh các khoảng thay đổi **ít đến mức nào**? Nếu nó chỉ thay đổi **cục bộ** quanh `value`, ta có thể **giữ sẵn** danh sách khoảng đã sắp xếp và chỉ sửa một chỗ nhỏ. Muốn vậy ta cần tìm nhanh "khoảng nằm ngay quanh `value`" — đó là việc của **binary search**.

## 4. Nền tảng: Binary search và module `bisect`

### 4.1 Binary search (tìm kiếm nhị phân) là gì?

Giả sử ta có list **đã sắp xếp** `starts = [1, 3, 7, 10, 15]` và muốn biết số `8` "đứng ở đâu" trong đó. Cách ngây thơ: đi từ đầu tới cuối, tốn `O(k)`. **Binary search** làm như khi tra từ điển giấy:

1. Nhìn phần tử **ở giữa** (`7`). `8 > 7` → chắc chắn chỗ của `8` nằm ở **nửa phải**, bỏ cả nửa trái.
2. Nhìn giữa nửa phải (`10` hoặc `15`), so sánh, lại bỏ một nửa…

Mỗi bước bỏ đi một nửa → sau khoảng `log2(k)` bước là xong. Với `k = 10^4`, `log2(10^4) ≈ 14` bước thay vì 10000 bước. Điều kiện bắt buộc: list phải **đã sắp xếp**.

### 4.2 Module `bisect` của thư viện chuẩn

Python đã cài sẵn binary search trong module `bisect`. Hai hàm ta cần, với list `a` đã sắp xếp:

- `bisect.bisect_left(a, x)`: vị trí **đầu tiên** có thể chèn `x` mà list vẫn sắp xếp = số phần tử **`< x`**.
- `bisect.bisect_right(a, x)`: vị trí **sau cùng** có thể chèn `x` = số phần tử **`<= x`**.

Hai hàm chỉ khác nhau khi `x` **đã có trong list**:

| `a = [1, 3, 7]` | `x = 0` | `x = 2` | `x = 3` | `x = 5` | `x = 7` | `x = 9` |
|---|---:|---:|---:|---:|---:|---:|
| `bisect_left(a, x)` | 0 | 1 | **1** | 2 | **2** | 3 |
| `bisect_right(a, x)` | 0 | 1 | **2** | 2 | **3** | 3 |

Với `x = 3`: có 1 số `< 3` (là `1`) nhưng có 2 số `<= 3` (là `1, 3`). Hai hàm đều tốn `O(log k)`.

In [9]:
import bisect

a = [1, 3, 7]
for x in [0, 2, 3, 5, 7, 9]:
    print(f"x={x}: bisect_left={bisect.bisect_left(a, x)}, bisect_right={bisect.bisect_right(a, x)}")

x=0: bisect_left=0, bisect_right=0
x=2: bisect_left=1, bisect_right=1
x=3: bisect_left=1, bisect_right=2
x=5: bisect_left=2, bisect_right=2
x=7: bisect_left=2, bisect_right=3
x=9: bisect_left=3, bisect_right=3


### 4.3 Mẹo cốt lõi: `i = bisect_right(starts, value) - 1` tìm "khoảng bên trái"

Giả sử ta đang giữ các khoảng **đã sắp xếp** theo `start`, tách thành 2 list song song:

```text
starts = [1, 6, 10]
ends   = [3, 7, 12]     # tức các khoảng [1,3], [6,7], [10,12]
```

Khi một số `value` tới, ta muốn biết: **khoảng nào có `start` lớn nhất mà vẫn `<= value`?** Gọi đó là **khoảng trái** `L` (left). Khoảng ngay sau nó (nếu có) là **khoảng phải** `R` — khoảng đầu tiên có `start > value`.

- `bisect_right(starts, value)` = số khoảng có `start <= value`.
- Trừ đi 1 → chỉ số của khoảng **cuối cùng** trong nhóm đó, chính là `L`. Khoảng phải `R` có chỉ số `i + 1`.
- Nếu `i == -1`: **không có** khoảng nào bắt đầu `<= value` → `value` nhỏ hơn mọi khoảng, không có `L`.
- Nếu `i + 1 == len(starts)`: không có `R`.

Vì sao dùng `bisect_right` mà **không** dùng `bisect_left`? Khi `value` **bằng đúng** một `start` (ví dụ `value = 6`), khoảng `[6, 7]` chứa `value` nên nó phải là `L`. `bisect_right` đếm cả số `== 6` nên trả `2`, trừ 1 được `1` → đúng `[6, 7]`. Còn `bisect_left` trả `1`, trừ 1 được `0` → nhầm sang `[1, 3]`. (Lỗi này có demo riêng ở mục lỗi thường gặp.)

In [10]:
starts = [1, 6, 10]
ends = [3, 7, 12]


def left_right(value):
    i = bisect.bisect_right(starts, value) - 1
    L = [starts[i], ends[i]] if i >= 0 else None
    R = [starts[i + 1], ends[i + 1]] if i + 1 < len(starts) else None
    return i, L, R


for value in [0, 2, 5, 6, 8, 13]:
    i, L, R = left_right(value)
    print(f"value={value:2d}: i={i:2d}  khoảng trái L={L}  khoảng phải R={R}")

value= 0: i=-1  khoảng trái L=None  khoảng phải R=[1, 3]
value= 2: i= 0  khoảng trái L=[1, 3]  khoảng phải R=[6, 7]
value= 5: i= 0  khoảng trái L=[1, 3]  khoảng phải R=[6, 7]
value= 6: i= 1  khoảng trái L=[6, 7]  khoảng phải R=[10, 12]
value= 8: i= 1  khoảng trái L=[6, 7]  khoảng phải R=[10, 12]
value=13: i= 2  khoảng trái L=[10, 12]  khoảng phải R=None


### 4.4 Chèn / xóa giữa list tốn bao nhiêu?

Tìm vị trí thì nhanh (`O(log k)`), nhưng để **sửa** list ta còn cần:

- `lst.insert(j, x)`: chèn `x` vào vị trí `j`. Mọi phần tử từ `j` trở đi phải **dịch sang phải** 1 ô → `O(k - j)`, tệ nhất `O(k)`.
- `lst.pop(j)`: xóa phần tử ở vị trí `j`, các phần tử sau **dịch sang trái** → `O(k - j)`, tệ nhất `O(k)`.
- Gán `lst[j] = x` thì chỉ `O(1)`.

Trên thực tế việc "dịch" này là một lệnh chép bộ nhớ liền khối (memmove) rất nhanh trong C, nên với `k` vài chục nghìn vẫn chạy tốt. Nhưng về lý thuyết ta phải ghi rõ: **tệ nhất `O(k)`**.

In [11]:
lst = [10, 20, 40]
lst.insert(2, 30)        # 40 dịch sang phải
print(lst)               # [10, 20, 30, 40]
lst.insert(0, 5)         # tất cả dịch sang phải -> đắt nhất
print(lst)               # [5, 10, 20, 30, 40]
print(lst.pop(1), lst)   # xóa 10, các phần tử sau dịch trái
lst[0] = 7               # gán O(1), không dịch gì
print(lst)

[10, 20, 30, 40]
[5, 10, 20, 30, 40]
10 [5, 20, 30, 40]
[7, 20, 30, 40]


## 5. Bước đột phá — chỉ hai khoảng `L` và `R` có thể bị ảnh hưởng

Giả sử các khoảng đang có thỏa **bất biến** (invariant): sắp xếp theo `start`, và giữa hai khoảng kề nhau luôn có khe hở (`ends[j] + 1 < starts[j + 1]`). Thêm một số `value`:

- Mọi khoảng **đứng trước `L`** kết thúc ở `<= L.start - 2 < value - 1`, tức cách `value` **ít nhất 2** → không chứa `value`, cũng không chạm `value`.
- Mọi khoảng **đứng sau `R`** bắt đầu ở `>= R.end + 2 > value + 1` → cũng không chạm `value`.

Vậy **chỉ có `L` và `R`** có thể dính tới `value`. Có đúng **5 trường hợp**, nhìn trên trục số (dấu `#` là số đã có, `v` là số mới):

| # | Trường hợp | Điều kiện | Hình minh họa | Hành động |
|---:|---|---|---|---|
| 1 | **Nằm trong** `L` | `L tồn tại` và `value <= L.end` | `##(v)##` — `v` trùng một ô `#` đã có | Không làm gì (số trùng / đã được phủ) |
| 2 | **Nối cả hai** | `L.end + 1 == value` **và** `value + 1 == R.start` | `###v###` (hai khối hàn lại) | `L.end = R.end`, xóa `R` |
| 3 | **Nối trái** | chỉ `L.end + 1 == value` | `###v  ##` | `L.end = value` |
| 4 | **Nối phải** | chỉ `value + 1 == R.start` | `##  v###` | `R.start = value` |
| 5 | **Đứng riêng** | không dính ai | `##  v  ##` | Chèn khoảng mới `[value, value]` giữa `L` và `R` |

Ví dụ cụ thể với các khoảng `[1, 3], [6, 7]`:

- `value = 2` → TH1 (nằm trong `[1, 3]`).
- `value = 5` → `L = [1, 3]`, `R = [6, 7]`: `3 + 1 = 4 != 5` nhưng `5 + 1 == 6` → TH4 nối phải → `[1, 3], [5, 7]`.
- `value = 4` (nếu đang có `[1, 3], [5, 7]`) → `3 + 1 == 4` và `4 + 1 == 5` → TH2 nối cả hai → `[1, 7]`.
- `value = 8` → `L = [6, 7]`, không có `R` → `7 + 1 == 8` → TH3 nối trái → `[6, 8]`.
- `value = 10` → `L = [6, 7]`, không chạm → TH5 → thêm `[10, 10]`.

In [12]:
def classify(starts, ends, value):
    # Trả về tên trường hợp khi thêm value vào các khoảng (starts, ends) đang có
    i = bisect.bisect_right(starts, value) - 1
    if i >= 0 and value <= ends[i]:
        return "1. nằm trong"
    left_touch = i >= 0 and ends[i] + 1 == value
    right_touch = i + 1 < len(starts) and starts[i + 1] == value + 1
    if left_touch and right_touch:
        return "2. nối cả hai"
    if left_touch:
        return "3. nối trái"
    if right_touch:
        return "4. nối phải"
    return "5. đứng riêng"


S, E = [1, 6], [3, 7]          # các khoảng [1,3], [6,7]
for v in [2, 5, 8, 10, 0, 4]:
    print(f"value={v:2d} -> {classify(S, E, v)}")
print("Với [1,3],[5,7] thêm 4 ->", classify([1, 5], [3, 7], 4))

value= 2 -> 1. nằm trong
value= 5 -> 4. nối phải
value= 8 -> 3. nối trái
value=10 -> 5. đứng riêng
value= 0 -> 4. nối phải
value= 4 -> 3. nối trái
Với [1,3],[5,7] thêm 4 -> 2. nối cả hai


### 5.1 Thứ tự kiểm tra và vài chi tiết tinh tế

1. **Kiểm tra "nằm trong" đầu tiên.** Nếu `value` đã nằm trong `L` thì mọi phép gộp khác đều vô nghĩa. Đây cũng là chỗ xử lý **số trùng**: thêm `2` lần thứ hai thì `2` nằm trong khoảng chứa nó → bỏ qua. Không cần `set` riêng!
2. **Kiểm tra "nối cả hai" trước "nối trái" / "nối phải".** Nếu kiểm tra nối trái trước rồi `return`, ta sẽ kéo dài `L` thành `[1, 4]` nhưng quên rằng `[5, 7]` giờ đã liền kề → kết quả `[1, 4], [5, 7]` **sai** (vi phạm "giữa hai khoảng phải có khe hở").
3. **Vì sao không thể chạm khoảng nào ngoài `L`, `R`?** Đã chứng minh ở trên nhờ bất biến khe hở. Đó cũng là lý do một lần `addNum` gộp **tối đa 2 khoảng** — không bao giờ phải gộp dây chuyền.
4. **Khi `value` nhỏ hơn mọi `start`** (`i = -1`): không có `L`. Phải viết `i >= 0 and ...` trước khi đọc `ends[i]`, vì trong Python `ends[-1]` **không báo lỗi** mà đọc **phần tử cuối**! Đây là cái bẫy rất nguy hiểm.

## 6. Cách 2 — Duy trì danh sách khoảng đã sắp xếp + `bisect` (cách chính)

Trạng thái của object: hai list song song `self.starts`, `self.ends` (khoảng thứ `j` là `[starts[j], ends[j]]`), luôn giữ bất biến:

- `starts` tăng dần;
- `ends[j] + 1 < starts[j + 1]` (có khe hở giữa hai khoảng kề nhau);
- hợp các khoảng = đúng tập các số đã thêm.

Vì sao tách 2 list thay vì một list các cặp `[s, e]`? Để gọi được `bisect_right(self.starts, value)` trực tiếp trên list `start` — dễ hiểu nhất cho người mới. (Với list các cặp ta phải dùng tham số `key=` của `bisect` từ Python 3.10, hoặc mẹo so sánh list theo thứ tự từ điển.)

`addNum(value)` gồm 3 bước:

1. **Tìm** `i = bisect_right(starts, value) - 1` (chỉ số của `L`, có thể là `-1`).
2. **Phân loại** đúng thứ tự: nằm trong → nối cả hai → nối trái → nối phải → đứng riêng.
3. **Sửa** list: gán (`O(1)`), hoặc `pop` một khoảng (TH2), hoặc `insert` một khoảng (TH5).

`getIntervals()`: ghép hai list thành `[[s, e], ...]` — **đã sắp xếp sẵn**, không cần sort.

In [13]:
class SummaryRangesBisect:
    # Cách 2: danh sách khoảng đã sắp xếp (2 list song song) + bisect
    def __init__(self):
        self.starts = []
        self.ends = []

    def addNum(self, value):
        starts, ends = self.starts, self.ends
        i = bisect.bisect_right(starts, value) - 1          # chỉ số khoảng trái L (-1 nếu không có)

        if i >= 0 and value <= ends[i]:                      # TH1: nằm trong L
            return

        left_touch = i >= 0 and ends[i] + 1 == value
        right_touch = i + 1 < len(starts) and starts[i + 1] == value + 1

        if left_touch and right_touch:                       # TH2: hàn L và R
            ends[i] = ends[i + 1]
            starts.pop(i + 1)
            ends.pop(i + 1)
        elif left_touch:                                     # TH3: kéo dài L sang phải
            ends[i] = value
        elif right_touch:                                    # TH4: kéo dài R sang trái
            starts[i + 1] = value
        else:                                                # TH5: khoảng mới giữa L và R
            starts.insert(i + 1, value)
            ends.insert(i + 1, value)

    def getIntervals(self):
        return [[s, e] for s, e in zip(self.starts, self.ends)]


out = run_commands(SummaryRangesBisect, EX1_OPS, EX1_ARGS)
print(show(out))
assert out == EX1_OUT
print("Cách 2 PASS Example 1")

[null, null, [[1, 1]], null, [[1, 1], [3, 3]], null, [[1, 1], [3, 3], [7, 7]], null, [[1, 3], [7, 7]], null, [[1, 3], [6, 7]]]
Cách 2 PASS Example 1


### 6.1 Trace chuỗi `1, 3, 7, 2, 6` (Example 1) rồi thêm `8, 2, 4, 5`

Năm số đầu là Example 1. Ta nối thêm `8, 2, 4, 5` để **đủ cả 5 trường hợp** và kết thúc bằng một cú "hàn" lớn.

| Bước | `value` | `starts` trước | `i` | `L` | `R` | Trường hợp | Khoảng sau |
|---:|---:|---|---:|---|---|---|---|
| 1 | 1 | `[]` | -1 | — | — | 5. đứng riêng | `[[1,1]]` |
| 2 | 3 | `[1]` | 0 | `[1,1]` | — | 5. đứng riêng (`1+1 != 3`) | `[[1,1],[3,3]]` |
| 3 | 7 | `[1,3]` | 1 | `[3,3]` | — | 5. đứng riêng | `[[1,1],[3,3],[7,7]]` |
| 4 | 2 | `[1,3,7]` | 0 | `[1,1]` | `[3,3]` | 2. nối cả hai (`1+1 == 2`, `2+1 == 3`) | `[[1,3],[7,7]]` |
| 5 | 6 | `[1,7]` | 0 | `[1,3]` | `[7,7]` | 4. nối phải (`6+1 == 7`) | `[[1,3],[6,7]]` |
| 6 | 8 | `[1,6]` | 1 | `[6,7]` | — | 3. nối trái (`7+1 == 8`) | `[[1,3],[6,8]]` |
| 7 | 2 | `[1,6]` | 0 | `[1,3]` | `[6,8]` | 1. nằm trong (`2 <= 3`) | `[[1,3],[6,8]]` |
| 8 | 4 | `[1,6]` | 0 | `[1,3]` | `[6,8]` | 3. nối trái (`3+1 == 4`, nhưng `4+1 != 6`) | `[[1,4],[6,8]]` |
| 9 | 5 | `[1,6]` | 0 | `[1,4]` | `[6,8]` | 2. nối cả hai (`4+1 == 5`, `5+1 == 6`) | `[[1,8]]` |

Hãy để ý bước 4: số `2` chạm **cả hai** khoảng `[1,1]` và `[3,3]`, nên số khoảng giảm từ 3 xuống 2. Bước 7: thêm lại `2` không làm gì cả. Bước 9: cả stream gộp còn **một** khoảng `[1, 8]` dù đã nhận 9 lần `addNum`.

Hàm `trace_bisect` dưới đây in đúng các cột trên từ code thật.

In [14]:
def trace_bisect(values):
    obj = SummaryRangesBisect()
    print(f"{'bước':>4} {'value':>5} {'i':>3}  {'L':8} {'R':8} {'trường hợp':16} khoảng sau")
    for step, value in enumerate(values, 1):
        st, en = obj.starts, obj.ends
        i = bisect.bisect_right(st, value) - 1
        L = f"[{st[i]},{en[i]}]" if i >= 0 else "-"
        R = f"[{st[i + 1]},{en[i + 1]}]" if i + 1 < len(st) else "-"
        case = classify(st, en, value)                  # phân loại TRƯỚC khi sửa
        obj.addNum(value)
        print(f"{step:>4} {value:>5} {i:>3}  {L:8} {R:8} {case:16} {obj.getIntervals()}")
    return obj.getIntervals()


trace_bisect([1, 3, 7, 2, 6, 8, 2, 4, 5])

bước value   i  L        R        trường hợp       khoảng sau
   1     1  -1  -        -        5. đứng riêng    [[1, 1]]
   2     3   0  [1,1]    -        5. đứng riêng    [[1, 1], [3, 3]]
   3     7   1  [3,3]    -        5. đứng riêng    [[1, 1], [3, 3], [7, 7]]
   4     2   0  [1,1]    [3,3]    2. nối cả hai    [[1, 3], [7, 7]]
   5     6   0  [1,3]    [7,7]    4. nối phải      [[1, 3], [6, 7]]
   6     8   1  [6,7]    -        3. nối trái      [[1, 3], [6, 8]]
   7     2   0  [1,3]    [6,8]    1. nằm trong     [[1, 3], [6, 8]]
   8     4   0  [1,3]    [6,8]    3. nối trái      [[1, 4], [6, 8]]
   9     5   0  [1,4]    [6,8]    2. nối cả hai    [[1, 8]]


[[1, 8]]

### 6.2 Vì sao cách 2 luôn đúng?

Ta chứng minh bằng **bất biến** (invariant) — một tính chất đúng **trước và sau mọi lần** `addNum`:

> **(I1)** `starts` tăng dần; **(I2)** `ends[j] + 1 < starts[j + 1]` với mọi `j` (có khe hở); **(I3)** hợp các khoảng = đúng tập các số đã thêm.

- **Ban đầu** hai list rỗng → cả 3 tính chất đúng hiển nhiên.
- **Giả sử đúng trước khi thêm `value`**, xét 5 trường hợp:
  - TH1: `value` đã được phủ → tập số không đổi, không sửa gì → vẫn đúng.
  - TH3 (nối trái): `L.end` tăng thêm 1 thành `value`. Khe hở với `R` vẫn còn vì **không** rơi vào TH2: `value + 1 != R.start` và `value < R.start` nên `value + 1 < R.start`. (I3) đúng vì ta phủ thêm đúng số `value`.
  - TH4 (nối phải): đối xứng TH3.
  - TH2 (nối cả hai): `[L.start, L.end] ∪ {value} ∪ [R.start, R.end] = [L.start, R.end]` vì ba mảnh nối liền. Khe hở với các khoảng xa hơn giữ nguyên từ trước.
  - TH5 (đứng riêng): `value` cách `L.end` ít nhất 2 và cách `R.start` ít nhất 2 → khoảng mới `[value, value]` có khe hở hai bên. Chèn ở `i + 1` giữ đúng thứ tự vì `L.start <= value < R.start`.
- Mục 5 đã chỉ ra **không** có khoảng nào ngoài `L`, `R` có thể chạm `value`, nên 5 trường hợp là **đủ**, không sót.

Từ (I1)–(I3), `getIntervals` trả về đúng các khoảng **rời nhau, gọn nhất, sắp xếp theo start** — chính là điều đề yêu cầu.

Cell dưới **kiểm chứng bằng máy**: sau mỗi `addNum` ngẫu nhiên, kiểm tra cả 3 bất biến và so với cách 1.

In [15]:
import random


def check_invariant(starts, ends, seen):
    for j in range(len(starts)):
        assert starts[j] <= ends[j]                            # khoảng hợp lệ
        if j + 1 < len(starts):
            assert starts[j] < starts[j + 1]                   # (I1)
            assert ends[j] + 1 < starts[j + 1]                 # (I2) có khe hở
    covered = {x for s, e in zip(starts, ends) for x in range(s, e + 1)}
    assert covered == seen                                     # (I3)


random.seed(352)
for trial in range(300):
    obj = SummaryRangesBisect()
    seen = set()
    hi = random.choice([5, 20, 60])                            # miền nhỏ -> nhiều gộp
    for _ in range(random.randint(1, 50)):
        v = random.randint(0, hi)
        obj.addNum(v)
        seen.add(v)
        check_invariant(obj.starts, obj.ends, seen)
    assert obj.getIntervals() == compress_sorted(sorted(seen))
print("300 chuỗi ngẫu nhiên: bất biến (I1)(I2)(I3) luôn đúng, kết quả khớp cách 1")

300 chuỗi ngẫu nhiên: bất biến (I1)(I2)(I3) luôn đúng, kết quả khớp cách 1


### 6.3 Độ phức tạp cách 2 (theo từng thao tác)

Gọi `k` là số khoảng hiện có.

| Thao tác | Time | Giải thích |
|---|---|---|
| Tìm `L` bằng `bisect_right` | `O(log k)` | binary search trên `starts` |
| TH1, TH3, TH4 | `O(1)` | chỉ so sánh / gán một ô |
| TH2 (`pop`), TH5 (`insert`) | `O(k)` tệ nhất | dịch các phần tử phía sau (memmove, rất nhanh) |
| ⇒ `addNum` | `O(log k)` khi không chèn/xóa, `O(k)` tệ nhất | |
| `getIntervals` | `O(k)` | chỉ ghép 2 list, **không sort** |
| Bộ nhớ | `O(k)` | chỉ lưu khoảng, **không lưu từng số** |

So với cách 1: `getIntervals` giảm từ `O(n log n)` xuống `O(k)`, bộ nhớ từ `O(n)` xuống `O(k)`. Đúng tinh thần câu follow-up: khi gộp nhiều thì `k` nhỏ, mọi thứ đều rẻ. Tổng cho cả `q` lời gọi: `O(q · k)` tệ nhất, thực tế với `k <= 5001` (số khoảng tối đa khi value trong `0..10^4`, vì giữa các khoảng phải có khe hở) chạy rất nhanh.

## 7. Code nộp LeetCode

Chuyển cách 2 sang đúng tên class / method / chữ ký của đề. Trên LeetCode, `List` đã được import sẵn; trong notebook ta tự `from typing import List`. Module `bisect` thuộc thư viện chuẩn nên dùng thoải mái.

In [16]:
from typing import List
import bisect


class SummaryRanges:

    def __init__(self):
        self.starts = []          # starts[j], ends[j] là khoảng thứ j; luôn sắp xếp, có khe hở
        self.ends = []

    def addNum(self, value: int) -> None:
        starts, ends = self.starts, self.ends
        i = bisect.bisect_right(starts, value) - 1   # khoảng trái: start lớn nhất <= value

        if i >= 0 and value <= ends[i]:               # nằm trong (kể cả số trùng)
            return

        left_touch = i >= 0 and ends[i] + 1 == value
        right_touch = i + 1 < len(starts) and starts[i + 1] == value + 1

        if left_touch and right_touch:                # nối cả hai
            ends[i] = ends[i + 1]
            starts.pop(i + 1)
            ends.pop(i + 1)
        elif left_touch:                              # nối trái
            ends[i] = value
        elif right_touch:                             # nối phải
            starts[i + 1] = value
        else:                                         # đứng riêng
            starts.insert(i + 1, value)
            ends.insert(i + 1, value)

    def getIntervals(self) -> List[List[int]]:
        return [[s, e] for s, e in zip(self.starts, self.ends)]


# Your SummaryRanges object will be instantiated and called as such:
# obj = SummaryRanges()
# obj.addNum(value)
# param_2 = obj.getIntervals()

### 7.1 Bộ test cho `SummaryRanges`

Ma trận test:

1. **Example 1** đúng nguyên văn input của LeetCode.
2. **Biên nhỏ nhất**: chỉ 1 số, gọi `getIntervals` ngay; gọi `getIntervals` hai lần liên tiếp (không được làm hỏng trạng thái).
3. **Biên giá trị**: `0` và `10^4`.
4. **Số trùng**: thêm cùng một số nhiều lần; thêm số nằm **giữa** một khoảng.
5. **Tăng dần / giảm dần liên tục**: toàn nối trái / toàn nối phải → còn đúng 1 khoảng.
6. **Hàn lớn**: các số lẻ trước, số chẵn sau → mỗi số chẵn hàn hai khoảng.
7. **Bẫy `i = -1`**: một khoảng lớn rồi thêm số nhỏ hơn mọi khoảng.

Hàm `check_tokens(cls, tokens, expected_last)` chạy chuỗi lệnh dựng bằng `build_commands` và so kết quả `getIntervals` **cuối cùng**.

In [17]:
def check_tokens(cls, tokens, expected_last, name=""):
    ops, args = build_commands(tokens + ["?"])
    got = run_commands(cls, ops, args)[-1]
    assert got == expected_last, (name, got, expected_last)
    return got


def run_basic_tests(cls):
    # 1) Example 1 nguyên văn
    assert run_commands(cls, EX1_OPS, EX1_ARGS) == EX1_OUT
    # 2) một số; hỏi hai lần liên tiếp
    assert run_commands(cls, ["SummaryRanges", "addNum", "getIntervals", "getIntervals"],
                        [[], [5], [], []]) == [None, None, [[5, 5]], [[5, 5]]]
    # 3) biên giá trị
    check_tokens(cls, [0, 10000, 1, 9999], [[0, 1], [9999, 10000]], "biên")
    # 4) số trùng, số nằm giữa khoảng
    check_tokens(cls, [3, 3, 3], [[3, 3]], "trùng")
    check_tokens(cls, [1, 2, 3, 4, 5, 3, 1, 5], [[1, 5]], "nằm giữa")
    # 5) tăng dần / giảm dần
    check_tokens(cls, list(range(10)), [[0, 9]], "tăng dần")
    check_tokens(cls, list(range(9, -1, -1)), [[0, 9]], "giảm dần")
    # 6) hàn lớn: lẻ trước, chẵn sau
    check_tokens(cls, [1, 3, 5, 7, 9], [[1, 1], [3, 3], [5, 5], [7, 7], [9, 9]], "lẻ")
    check_tokens(cls, [1, 3, 5, 7, 9, 2, 4, 6, 8], [[1, 9]], "hàn")
    # 7) bẫy i = -1
    check_tokens(cls, [3, 4, 5, 6, 7, 8, 9, 1], [[1, 1], [3, 9]], "i=-1")
    check_tokens(cls, [3, 4, 5, 6, 7, 8, 9, 2], [[2, 9]], "i=-1 nối phải")
    return True


print(show(run_commands(SummaryRanges, EX1_OPS, EX1_ARGS)))
run_basic_tests(SummaryRanges)
print("SummaryRanges PASS toàn bộ test cơ bản")

[null, null, [[1, 1]], null, [[1, 1], [3, 3]], null, [[1, 1], [3, 3], [7, 7]], null, [[1, 3], [7, 7]], null, [[1, 3], [6, 7]]]
SummaryRanges PASS toàn bộ test cơ bản


## 8. Cách 3 — Hai dict "hai đầu": gộp `O(1)`, sắp xếp khi hỏi

### 8.1 Động lực

Ở cách 2, `addNum` vẫn có thể tốn `O(k)` vì phải `insert` / `pop` giữa list. Constraints lại nói `getIntervals` bị gọi **rất ít** (`<= 100`) so với `addNum` (gần `3·10^4`). Vậy có cách nào làm `addNum` thật nhẹ — `O(1)` — và chấp nhận `getIntervals` nặng hơn không?

Quan sát: trong 5 trường hợp, ta chỉ cần biết hai điều:

- Có khoảng nào **kết thúc đúng tại `value - 1`** không? (để nối trái)
- Có khoảng nào **bắt đầu đúng tại `value + 1`** không? (để nối phải)

Đây là câu hỏi "tra đúng một khóa" — việc sở trường của **dict** (bảng băm), trung bình `O(1)`, không cần thứ tự. Ta giữ hai dict:

- `start_to_end[s] = e`: tra từ đầu trái ra đầu phải.
- `end_to_start[e] = s`: tra từ đầu phải ra đầu trái.

Còn TH1 "nằm trong" thì sao? Dict chỉ biết hai **đầu mút**, không biết số `2` nằm giữa `[1, 3]`. Vì vậy cần thêm một `set` `seen` để nhận ra số đã có. Cái giá: bộ nhớ quay về `O(n)`.

### 8.2 Ôn nhanh các thao tác dict cần dùng

In [18]:
d = {}
d[1] = 3                    # thêm / ghi đè khóa 1
d[7] = 7
print(d, 1 in d, 2 in d)    # {1: 3, 7: 7} True False
e = d.pop(7)                # lấy giá trị RA và XÓA khóa 7
print(e, d)                 # 7 {1: 3}
d[6] = 7
print(d)                    # dict nhớ THỨ TỰ CHÈN, không phải thứ tự khóa: {1: 3, 6: 7}
d[0] = 0
print(d, "->", sorted(d.items()))   # muốn theo thứ tự khóa phải sorted

{1: 3, 7: 7} True False
7 {1: 3}
{1: 3, 6: 7}
{1: 3, 6: 7, 0: 0} -> [(0, 0), (1, 3), (6, 7)]


### 8.3 Bốn trường hợp với hai dict

Sau khi loại TH1 bằng `seen`:

| Trường hợp | Điều kiện | Cập nhật |
|---|---|---|
| Nối cả hai | `value - 1 in end_to_start` và `value + 1 in start_to_end` | `s = end_to_start.pop(value - 1)`, `e = start_to_end.pop(value + 1)`; xóa luôn `start_to_end[s]` cũ và `end_to_start[e]` cũ; ghi khoảng mới `[s, e]` |
| Nối trái | chỉ `value - 1 in end_to_start` | `s = end_to_start.pop(value - 1)`; khoảng thành `[s, value]` |
| Nối phải | chỉ `value + 1 in start_to_end` | `e = start_to_end.pop(value + 1)`; khoảng thành `[value, e]` |
| Đứng riêng | không | khoảng mới `[value, value]` |

Mẹo viết gọn: gọi `s` là đầu trái mới, `e` là đầu phải mới (mặc định đều là `value`); nối trái thì `s` lấy từ khoảng trái, nối phải thì `e` lấy từ khoảng phải. **Xóa hết các khóa cũ** rồi ghi `start_to_end[s] = e`, `end_to_start[e] = s`. Bốn trường hợp gộp thành một đoạn code!

`getIntervals`: `sorted(start_to_end.items())` → `O(k log k)`.

In [19]:
class SummaryRangesTwoMaps:
    # Cách 3: set + hai dict hai đầu; addNum O(1) trung bình, getIntervals O(k log k)
    def __init__(self):
        self.seen = set()
        self.start_to_end = {}
        self.end_to_start = {}

    def addNum(self, value):
        if value in self.seen:                    # TH1: đã có (nằm trong một khoảng)
            return
        self.seen.add(value)
        s = e = value
        if value - 1 in self.end_to_start:        # có khoảng [s0, value-1] -> nối trái
            s = self.end_to_start.pop(value - 1)
            del self.start_to_end[s]              # khóa cũ của khoảng trái
        if value + 1 in self.start_to_end:        # có khoảng [value+1, e0] -> nối phải
            e = self.start_to_end.pop(value + 1)
            del self.end_to_start[e]              # khóa cũ của khoảng phải
        self.start_to_end[s] = e                  # ghi khoảng mới [s, e]
        self.end_to_start[e] = s

    def getIntervals(self):
        return [[s, e] for s, e in sorted(self.start_to_end.items())]


out = run_commands(SummaryRangesTwoMaps, EX1_OPS, EX1_ARGS)
print(show(out))
assert out == EX1_OUT
run_basic_tests(SummaryRangesTwoMaps)
print("Cách 3 PASS Example 1 + test cơ bản")

[null, null, [[1, 1]], null, [[1, 1], [3, 3]], null, [[1, 1], [3, 3], [7, 7]], null, [[1, 3], [7, 7]], null, [[1, 3], [6, 7]]]
Cách 3 PASS Example 1 + test cơ bản


### 8.4 Trace hai dict với chuỗi `1, 3, 7, 2, 6`

| `value` | `value-1` là đầu phải? | `value+1` là đầu trái? | Khoảng mới `[s, e]` | `start_to_end` sau | `end_to_start` sau |
|---:|---|---|---|---|---|
| 1 | `0`: không | `2`: không | `[1, 1]` | `{1: 1}` | `{1: 1}` |
| 3 | `2`: không | `4`: không | `[3, 3]` | `{1: 1, 3: 3}` | `{1: 1, 3: 3}` |
| 7 | `6`: không | `8`: không | `[7, 7]` | `{1: 1, 3: 3, 7: 7}` | `{1: 1, 3: 3, 7: 7}` |
| 2 | `1`: **có** (khoảng `[1,1]`) | `3`: **có** (khoảng `[3,3]`) | `[1, 3]` | `{7: 7, 1: 3}` | `{7: 7, 3: 1}` |
| 6 | `5`: không | `7`: **có** (khoảng `[7,7]`) | `[6, 7]` | `{1: 3, 6: 7}` | `{3: 1, 7: 6}` |

Ở bước `value = 2`: lấy `s = end_to_start.pop(1) = 1` và xóa `start_to_end[1]`; lấy `e = start_to_end.pop(3) = 3` và xóa `end_to_start[3]`; cuối cùng ghi `start_to_end[1] = 3`, `end_to_start[3] = 1`. Để ý: khóa `1` bị xóa rồi ghi lại nên nó chuyển xuống **cuối** dict (`{7: 7, 1: 3}`) — thứ tự khóa trong dict là **thứ tự chèn**, không phải thứ tự `start`. Chính vì thế `getIntervals` phải `sorted`.

In [20]:
def trace_two_maps(values):
    obj = SummaryRangesTwoMaps()
    for value in values:
        lt = value - 1 in obj.end_to_start
        rt = value + 1 in obj.start_to_end
        obj.addNum(value)
        print(f"value={value}: nối trái={str(lt):5s} nối phải={str(rt):5s} "
              f"start_to_end={obj.start_to_end}  end_to_start={obj.end_to_start}")
    print("getIntervals ->", obj.getIntervals())


trace_two_maps([1, 3, 7, 2, 6])

value=1: nối trái=False nối phải=False start_to_end={1: 1}  end_to_start={1: 1}
value=3: nối trái=False nối phải=False start_to_end={1: 1, 3: 3}  end_to_start={1: 1, 3: 3}
value=7: nối trái=False nối phải=False start_to_end={1: 1, 3: 3, 7: 7}  end_to_start={1: 1, 3: 3, 7: 7}
value=2: nối trái=True  nối phải=True  start_to_end={7: 7, 1: 3}  end_to_start={7: 7, 3: 1}
value=6: nối trái=False nối phải=True  start_to_end={1: 3, 6: 7}  end_to_start={3: 1, 7: 6}
getIntervals -> [[1, 3], [6, 7]]


### 8.5 Độ phức tạp cách 3

| Thao tác | Time | Ghi chú |
|---|---|---|
| `addNum` | `O(1)` trung bình | vài phép tra / xóa / ghi dict và set |
| `getIntervals` | `O(k log k)` | sắp xếp `k` khoảng theo `start` |
| Bộ nhớ | `O(n + k)` = `O(n)` | `seen` lưu **mọi** số phân biệt |

Với constraints của đề (`addNum` nhiều, `getIntervals <= 100`), đây là cách rất hợp: tổng chi phí khoảng `O(q + g · k log k)`.

Nhưng với câu **follow-up** ("gộp rất nhiều, số khoảng nhỏ"), `seen` vẫn phình theo `n`. Có thể bỏ `seen` không? Muốn biết "`value` có nằm **giữa** một khoảng không" mà không lưu từng số, ta phải tìm được khoảng có `start` lớn nhất `<= value` — tức là cần một cấu trúc **có thứ tự**. Đó là phần tiếp theo.

## 9. Cách 4 (follow-up) — Cây cân bằng / `SortedDict`

### 9.1 Ta cần một "map có thứ tự"

Đọc lại cách 2: thao tác thật sự cần là

- `floor_key(value)`: khóa (start) **lớn nhất `<= value`** → khoảng trái `L`;
- `higher_key(value)`: khóa **nhỏ nhất `> value`** → khoảng phải `R`;
- thêm / xóa / sửa một khoảng;
- duyệt mọi khoảng theo thứ tự `start`.

Một **cây nhị phân tìm kiếm cân bằng** (balanced BST: AVL, red-black tree...) làm **mọi** thao tác trên trong `O(log k)`, duyệt thứ tự trong `O(k)`. Đây là đáp án "sách giáo khoa" cho follow-up:

- Java: `TreeMap<Integer, Integer>` với `floorKey`, `higherKey`.
- C++: `std::map` với `upper_bound` rồi lùi 1.
- Python: **không có** cây cân bằng trong thư viện chuẩn. LeetCode có cài sẵn thư viện ngoài `sortedcontainers` với `SortedDict`, bên trong không phải cây mà là "list của các list nhỏ"; tài liệu của thư viện ghi thêm / xóa xấp xỉ `O(log k)` và thực tế nhanh ngang cây cân bằng — đủ dùng như một TreeMap.

Khi đó: `addNum` `O(log k)`, `getIntervals` `O(k)`, bộ nhớ `O(k)` — **không phụ thuộc `n`**. Khi gộp nhiều (`k` nhỏ) thì mọi thao tác gần như hằng số. Đó là câu trả lời cho follow-up.

Bản dùng `SortedDict` (chạy được trên LeetCode, **không** chạy trong notebook vì chỉ dùng thư viện chuẩn):

```python
from sortedcontainers import SortedDict

class SummaryRanges:
    def __init__(self):
        self.mp = SortedDict()                     # start -> end

    def addNum(self, value: int) -> None:
        mp = self.mp
        i = mp.bisect_right(value) - 1             # chỉ số của floor key
        if i >= 0 and mp.peekitem(i)[1] >= value:  # nằm trong
            return
        left = i >= 0 and mp.peekitem(i)[1] + 1 == value
        right = i + 1 < len(mp) and mp.peekitem(i + 1)[0] == value + 1
        if left and right:
            ls, _ = mp.peekitem(i)
            _, re = mp.popitem(i + 1)
            mp[ls] = re
        elif left:
            mp[mp.peekitem(i)[0]] = value
        elif right:
            _, re = mp.popitem(i + 1)
            mp[value] = re
        else:
            mp[value] = value

    def getIntervals(self) -> List[List[int]]:
        return [[s, e] for s, e in self.mp.items()]
```

### 9.2 Tự cài một "sorted map" mini để thấy rõ giao diện TreeMap

Để chạy được trong notebook chỉ với thư viện chuẩn, ta tự viết class `ListSortedMap` có **đúng giao diện** của một TreeMap (`floor_key`, `higher_key`, `get`, `set`, `remove`, `items`). Bên trong nó dùng list + `bisect` (nên thêm / xóa vẫn `O(k)` như cách 2) — nhưng điểm đáng học là: **thuật toán viết trên giao diện TreeMap**, sau này đổi ruột sang cây cân bằng thật là có ngay `O(log k)` mà không sửa một dòng logic nào.

In [21]:
class ListSortedMap:
    # Map có thứ tự theo khóa, giao diện giống TreeMap. Ruột: list + bisect.
    def __init__(self):
        self.keys = []
        self.vals = []

    def floor_key(self, x):            # khóa lớn nhất <= x, hoặc None
        i = bisect.bisect_right(self.keys, x) - 1
        return self.keys[i] if i >= 0 else None

    def higher_key(self, x):           # khóa nhỏ nhất > x, hoặc None
        i = bisect.bisect_right(self.keys, x)
        return self.keys[i] if i < len(self.keys) else None

    def get(self, key):
        return self.vals[bisect.bisect_left(self.keys, key)]

    def set(self, key, val):
        i = bisect.bisect_left(self.keys, key)
        if i < len(self.keys) and self.keys[i] == key:
            self.vals[i] = val
        else:
            self.keys.insert(i, key)
            self.vals.insert(i, val)

    def remove(self, key):
        i = bisect.bisect_left(self.keys, key)
        self.keys.pop(i)
        self.vals.pop(i)

    def items(self):
        return list(zip(self.keys, self.vals))


m = ListSortedMap()
for k_, v_ in [(6, 7), (1, 3), (10, 12)]:
    m.set(k_, v_)
print(m.items())                                  # luôn theo thứ tự khóa
print(m.floor_key(8), m.higher_key(8))            # 6 10
print(m.floor_key(0), m.higher_key(12))           # None None

[(1, 3), (6, 7), (10, 12)]
6 10
None None


In [22]:
class SummaryRangesTreeMap:
    # Cách 4: viết trên giao diện TreeMap (start -> end); không cần set seen, bộ nhớ O(k)
    def __init__(self):
        self.mp = ListSortedMap()

    def addNum(self, value):
        mp = self.mp
        ls = mp.floor_key(value)                  # start của khoảng trái L (hoặc None)
        rs = mp.higher_key(value)                 # start của khoảng phải R (hoặc None)
        if ls is not None and mp.get(ls) >= value:            # nằm trong L
            return
        left = ls is not None and mp.get(ls) + 1 == value
        right = rs is not None and rs == value + 1
        if left and right:                                    # nối cả hai
            mp.set(ls, mp.get(rs))
            mp.remove(rs)
        elif left:                                            # nối trái
            mp.set(ls, value)
        elif right:                                           # nối phải: đổi khóa rs -> value
            re = mp.get(rs)
            mp.remove(rs)
            mp.set(value, re)
        else:                                                 # đứng riêng
            mp.set(value, value)

    def getIntervals(self):
        return [[s, e] for s, e in self.mp.items()]


out = run_commands(SummaryRangesTreeMap, EX1_OPS, EX1_ARGS)
print(show(out))
assert out == EX1_OUT
run_basic_tests(SummaryRangesTreeMap)
print("Cách 4 PASS Example 1 + test cơ bản")

[null, null, [[1, 1]], null, [[1, 1], [3, 3]], null, [[1, 1], [3, 3], [7, 7]], null, [[1, 3], [7, 7]], null, [[1, 3], [6, 7]]]
Cách 4 PASS Example 1 + test cơ bản


### 9.3 Trả lời câu follow-up

> *Nếu có rất nhiều lần gộp và số khoảng nhỏ so với kích thước stream thì sao?*

- **Đừng lưu từng số.** Cách 1 (`set` / mảng đánh dấu) và cách 3 (`seen`) tốn bộ nhớ `O(n)` hoặc `O(V)`. Cách 2 / cách 4 chỉ lưu **khoảng** → `O(k)`. Khi gộp nhiều, `k << n`.
- **Số trùng / số nằm giữa khoảng** được nhận ra bằng chính khoảng trái `L` (`value <= L.end`), không cần `seen`.
- **Thời gian** `addNum` = `O(log k)` (cây cân bằng / `SortedDict`) hoặc `O(log k)` tìm + `O(k)` dịch list (cách 2). Khi `k` nhỏ, cả hai đều gần như hằng số; khi `k` lớn mà vẫn muốn bảo đảm lý thuyết thì dùng cây cân bằng.
- **`getIntervals`** = `O(k)`, không cần sắp xếp lại.

Tóm lại: *"giữ các khoảng trong một map có thứ tự `start → end`; tìm floor/higher của `value`; xử lý 5 trường hợp"*.

## 10. Test đối chiếu chéo trên chuỗi lệnh ngẫu nhiên

Ta có 6 cài đặt: cách 1 (`SummaryRangesSortSet`), biến thể mảng đánh dấu (`SummaryRangesBoolArray`), cách 2 (`SummaryRangesBisect`), bản nộp (`SummaryRanges`), cách 3 (`SummaryRangesTwoMaps`), cách 4 (`SummaryRangesTreeMap`). Sinh ngẫu nhiên (có `random.seed` cố định để chạy lại ra y hệt) nhiều chuỗi lệnh xen kẽ `addNum` / `getIntervals`, chạy **cùng** chuỗi trên cả 6 class và yêu cầu **toàn bộ** list output giống hệt nhau.

Để có đủ mọi trường hợp, miền giá trị được chọn ngẫu nhiên giữa **rất hẹp** (`0..10`, gộp liên tục, nhiều số trùng), **vừa** (`0..100`) và **cả miền** `0..10^4` (ít gộp).

In [23]:
ALL_CLASSES = [SummaryRangesSortSet, SummaryRangesBoolArray, SummaryRangesBisect,
               SummaryRanges, SummaryRangesTwoMaps, SummaryRangesTreeMap]


def random_commands(rng, length, hi):
    ops, args = ["SummaryRanges"], [[]]
    for _ in range(length):
        if rng.random() < 0.3:
            ops.append("getIntervals")
            args.append([])
        else:
            ops.append("addNum")
            args.append([rng.randint(0, hi)])
    ops.append("getIntervals")
    args.append([])
    return ops, args


rng = random.Random(2024)
trials = 0
for _ in range(500):
    hi = rng.choice([10, 100, 10**4])
    ops, args = random_commands(rng, rng.randint(1, 80), hi)
    expected = run_commands(SummaryRangesSortSet, ops, args)
    for cls in ALL_CLASSES[1:]:
        got = run_commands(cls, ops, args)
        assert got == expected, (cls.__name__, ops, args)
    trials += 1
print(f"{trials} chuỗi lệnh ngẫu nhiên: cả {len(ALL_CLASSES)} cài đặt cho output giống hệt nhau")

500 chuỗi lệnh ngẫu nhiên: cả 6 cài đặt cho output giống hệt nhau


### 10.1 Test cỡ lớn đúng giới hạn đề + đo thời gian

Hai kịch bản, mỗi kịch bản đúng `3·10^4` lời gọi, trong đó `100` lời gọi `getIntervals` rải đều:

- **Ngẫu nhiên** trên `0..10^4`: ít gộp hơn, `k` lớn (hàng trăm tới hàng nghìn khoảng trong suốt quá trình).
- **Nhiều gộp** (tinh thần follow-up): thêm mọi số của `0..10^4` theo một thứ tự xáo trộn, rồi lặp lại chính dãy đó (toàn số trùng) → cuối cùng chỉ còn 1 khoảng `[0, 10000]`.

Thời gian in ra phụ thuộc máy nên mỗi lần chạy sẽ hơi khác; điều cần quan sát là **tất cả đều nhanh** với constraints của đề và các output khớp nhau. (`SummaryRangesBisect` và `SummaryRanges` là **cùng một** thuật toán; nếu thời gian của chúng chênh nhau thì đó chỉ là nhiễu đo — bộ nhớ đệm, thứ tự chạy...)

In [24]:
import time


def big_scenario(values):
    ops, args = ["SummaryRanges"], [[]]
    gap = len(values) // 100
    for idx, v in enumerate(values):
        ops.append("addNum")
        args.append([v])
        if (idx + 1) % gap == 0 and ops.count("getIntervals") < 100:
            ops.append("getIntervals")
            args.append([])
    return ops, args


rng = random.Random(7)
random_vals = [rng.randint(0, 10**4) for _ in range(29900)]
merge_vals = list(range(10**4 + 1))
rng.shuffle(merge_vals)
merge_vals = (merge_vals * 3)[:29900]

for title, vals in [("ngẫu nhiên", random_vals), ("nhiều gộp", merge_vals)]:
    ops, args = big_scenario(vals)
    assert len(ops) - 1 == 30000 and ops.count("getIntervals") == 100
    ref = None
    print(f"Kịch bản {title}:")
    for cls in ALL_CLASSES:
        t0 = time.perf_counter()
        out = run_commands(cls, ops, args)
        dt = time.perf_counter() - t0
        ref = out if ref is None else ref
        assert out == ref
        print(f"  {cls.__name__:24s} {dt * 1000:7.1f} ms   số khoảng cuối = {len(out[-1])}")

Kịch bản ngẫu nhiên:
  SummaryRangesSortSet        99.4 ms   số khoảng cuối = 461


  SummaryRangesBoolArray      93.7 ms   số khoảng cuối = 461


  SummaryRangesBisect        125.8 ms   số khoảng cuối = 461
  SummaryRanges               55.9 ms   số khoảng cuối = 461


  SummaryRangesTwoMaps       115.8 ms   số khoảng cuối = 461


  SummaryRangesTreeMap        60.1 ms   số khoảng cuối = 461
Kịch bản nhiều gộp:
  SummaryRangesSortSet        50.9 ms   số khoảng cuối = 1


  SummaryRangesBoolArray      43.1 ms   số khoảng cuối = 1
  SummaryRangesBisect         17.2 ms   số khoảng cuối = 1
  SummaryRanges               15.8 ms   số khoảng cuối = 1
  SummaryRangesTwoMaps        54.4 ms   số khoảng cuối = 1


  SummaryRangesTreeMap        33.3 ms   số khoảng cuối = 1


## 11. Bảng so sánh các cách

`n` = số giá trị phân biệt, `k` = số khoảng, `V = 10^4 + 1`.

| Cách | Trạng thái lưu | `addNum` | `getIntervals` | Bộ nhớ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|---|
| 1. `set` + sort khi hỏi | mọi số | `O(1)` TB | `O(n log n)` | `O(n)` | dễ nghĩ, khó sai | làm lại từ đầu mỗi lần hỏi |
| 1b. Mảng đánh dấu | `V` ô True/False | `O(1)` | `O(V)` | `O(V)` | không cần sort | gắn chặt vào `value <= 10^4` |
| **2. List khoảng + `bisect`** | **chỉ khoảng** | `O(log k)` tìm + `O(k)` dịch tệ nhất | **`O(k)`** | **`O(k)`** | cân bằng, bộ nhớ nhỏ, chỉ thư viện chuẩn | `insert`/`pop` giữa list |
| 3. Hai dict hai đầu + `seen` | mọi số + 2 dict | **`O(1)` TB** | `O(k log k)` | `O(n)` | `addNum` nhẹ nhất, hợp với `g <= 100` | không trả lời được follow-up về bộ nhớ |
| 4. TreeMap / `SortedDict` | chỉ khoảng | `O(log k)` | `O(k)` | `O(k)` | tốt nhất cho follow-up | Python chuẩn không có cây cân bằng |

Chọn cách nào?

- **Phỏng vấn / nộp bài bằng Python thuần**: cách 2 — vừa đúng tinh thần follow-up, vừa không cần thư viện ngoài.
- **Rất nhiều `addNum`, rất ít `getIntervals`** và không quan tâm bộ nhớ: cách 3.
- **Java / C++** hoặc được dùng `sortedcontainers`: cách 4 là đáp án gọn nhất.

## 12. Các lỗi người mới hay gặp

Mỗi lỗi: **code sai → chạy cho thấy sai ở đâu → vì sao → cách sửa**. Các class sai đặt tên `...Wrong...` để không đè lên lời giải đúng.

### Lỗi 1: Quên trường hợp "nối cả hai"

Người mới hay viết `if nối trái ... elif nối phải ...` và nghĩ thế là đủ. Khi `value` chạm **cả hai** phía, code chỉ kéo dài `L`, còn `R` vẫn đứng riêng ngay sát bên → hai khoảng liền kề không được gộp.

In [25]:
class SummaryRangesWrongNoBoth(SummaryRangesBisect):
    def addNum(self, value):
        starts, ends = self.starts, self.ends
        i = bisect.bisect_right(starts, value) - 1
        if i >= 0 and value <= ends[i]:
            return
        if i >= 0 and ends[i] + 1 == value:                       # SAI: dừng ở đây, quên nhìn R
            ends[i] = value
        elif i + 1 < len(starts) and starts[i + 1] == value + 1:
            starts[i + 1] = value
        else:
            starts.insert(i + 1, value)
            ends.insert(i + 1, value)


tokens = [1, 3, 7, 2]
wrong = SummaryRangesWrongNoBoth()
for v in tokens:
    wrong.addNum(v)
print("Sai :", wrong.getIntervals())      # [[1, 2], [3, 3], [7, 7]] -> [1,2] và [3,3] liền kề!
print("Đúng:", check_tokens(SummaryRanges, tokens, [[1, 3], [7, 7]]))

Sai : [[1, 2], [3, 3], [7, 7]]
Đúng: [[1, 3], [7, 7]]


**Sửa:** tính cả hai cờ `left_touch`, `right_touch` **trước**, rồi kiểm tra `if left_touch and right_touch` **đầu tiên**. Nhớ rằng bất biến "giữa hai khoảng luôn có khe hở" phải đúng sau **mọi** lần `addNum`.

### Lỗi 2: Dùng `bisect_left` thay vì `bisect_right`

Khi `value` **bằng đúng** `start` của một khoảng (ví dụ thêm lại số `7` khi đang có `[7, 7]`), `bisect_left(starts, 7) - 1` chỉ về khoảng **trước** đó. Code tưởng `value` không nằm trong khoảng nào và chèn thêm `[7, 7]` → khoảng bị **lặp**.

In [26]:
class SummaryRangesWrongBisectLeft(SummaryRangesBisect):
    def addNum(self, value):
        starts, ends = self.starts, self.ends
        i = bisect.bisect_left(starts, value) - 1                   # SAI: nên là bisect_right
        if i >= 0 and value <= ends[i]:
            return
        left_touch = i >= 0 and ends[i] + 1 == value
        right_touch = i + 1 < len(starts) and starts[i + 1] == value + 1
        if left_touch and right_touch:
            ends[i] = ends[i + 1]
            starts.pop(i + 1)
            ends.pop(i + 1)
        elif left_touch:
            ends[i] = value
        elif right_touch:
            starts[i + 1] = value
        else:
            starts.insert(i + 1, value)
            ends.insert(i + 1, value)


tokens = [1, 2, 3, 7, 7]
wrong = SummaryRangesWrongBisectLeft()
for v in tokens:
    wrong.addNum(v)
print("Sai :", wrong.getIntervals())      # [[1, 3], [7, 7], [7, 7]]
print("Đúng:", check_tokens(SummaryRanges, tokens, [[1, 3], [7, 7]]))

Sai : [[1, 3], [7, 7], [7, 7]]
Đúng: [[1, 3], [7, 7]]


**Sửa:** `i = bisect_right(starts, value) - 1`. Câu cần nhớ: *"khoảng trái là khoảng có start **lớn nhất `<=` value** → `bisect_right` rồi trừ 1"*. `bisect_right` đếm cả các phần tử **bằng** `value`.

### Lỗi 3: Quên kiểm tra `i >= 0` — bẫy chỉ số âm của Python

Khi `value` nhỏ hơn mọi `start`, `i = -1`. Trong nhiều ngôn ngữ, `ends[-1]` sẽ báo lỗi ngay. Trong Python, `ends[-1]` là **phần tử cuối cùng** → code chạy "êm" nhưng **sai âm thầm**.

In [27]:
class SummaryRangesWrongNegIndex(SummaryRangesBisect):
    def addNum(self, value):
        starts, ends = self.starts, self.ends
        i = bisect.bisect_right(starts, value) - 1
        if starts and value <= ends[i]:                  # SAI: i có thể là -1 -> ends[-1] là khoảng CUỐI
            return
        SummaryRangesBisect.addNum(self, value)


tokens = [3, 4, 5, 6, 7, 8, 9, 1]
wrong = SummaryRangesWrongNegIndex()
for v in tokens:
    wrong.addNum(v)
print("Sai :", wrong.getIntervals())      # [[3, 9]] -> số 1 bị "nuốt" mất vì 1 <= ends[-1] = 9
print("Đúng:", check_tokens(SummaryRanges, tokens, [[1, 1], [3, 9]]))

Sai : [[3, 9]]
Đúng: [[1, 1], [3, 9]]


**Sửa:** luôn viết `i >= 0 and ...` trước khi đọc `starts[i]` / `ends[i]`, và `i + 1 < len(starts) and ...` trước khi đọc `starts[i + 1]`. Toán tử `and` **dừng sớm** (short-circuit): vế trái `False` thì vế phải không được chạy, nên không đọc nhầm chỉ số.

### Lỗi 4: Ba lỗi kinh điển của cách hai dict

- **(4a) Quên `seen`**: thêm lại số `2` nằm **giữa** `[1, 3]` → `1` không phải đầu phải, `3` không phải đầu trái → code tưởng đứng riêng và tạo `[2, 2]` chồng lên `[1, 3]`.
- **(4b) Dùng `d[key]` thay vì `d.pop(key)`**: khóa cũ của khoảng bị gộp vẫn nằm trong dict → `getIntervals` trả ra khoảng "ma".
- **(4c) Quên `sorted`** trong `getIntervals`: dict giữ **thứ tự chèn**, không phải thứ tự `start`.

In [28]:
class WrongNoSeen(SummaryRangesTwoMaps):
    def addNum(self, value):                                   # (4a) bỏ dòng kiểm tra seen
        s = e = value
        if value - 1 in self.end_to_start:
            s = self.end_to_start.pop(value - 1)
            del self.start_to_end[s]
        if value + 1 in self.start_to_end:
            e = self.start_to_end.pop(value + 1)
            del self.end_to_start[e]
        self.start_to_end[s] = e
        self.end_to_start[e] = s


class WrongNoPop(SummaryRangesTwoMaps):
    def addNum(self, value):                                   # (4b) đọc nhưng không xóa khóa cũ
        if value in self.seen:
            return
        self.seen.add(value)
        s = e = value
        if value - 1 in self.end_to_start:
            s = self.end_to_start[value - 1]
        if value + 1 in self.start_to_end:
            e = self.start_to_end[value + 1]
        self.start_to_end[s] = e
        self.end_to_start[e] = s


class WrongNoSort(SummaryRangesTwoMaps):
    def getIntervals(self):                                    # (4c) quên sorted
        return [[s, e] for s, e in self.start_to_end.items()]


for cls, tokens, good in [(WrongNoSeen, [1, 2, 3, 2], [[1, 3]]),
                          (WrongNoPop, [3, 2], [[2, 3]]),
                          (WrongNoSort, [7, 1], [[1, 1], [7, 7]])]:
    obj = cls()
    for v in tokens:
        obj.addNum(v)
    print(f"{cls.__name__:12s} tokens={tokens}: sai = {obj.getIntervals()}, đúng = {good}")
    assert check_tokens(SummaryRangesTwoMaps, tokens, good) == good

WrongNoSeen  tokens=[1, 2, 3, 2]: sai = [[1, 3], [2, 2]], đúng = [[1, 3]]
WrongNoPop   tokens=[3, 2]: sai = [[2, 3], [3, 3]], đúng = [[2, 3]]
WrongNoSort  tokens=[7, 1]: sai = [[7, 7], [1, 1]], đúng = [[1, 1], [7, 7]]


**Sửa:** giữ đủ 3 thứ trong `SummaryRangesTwoMaps`: kiểm tra `value in self.seen` đầu tiên; lấy khóa cũ ra bằng `pop`; `getIntervals` luôn `sorted(...)`.

### Bảng tóm tắt lỗi

| # | Lỗi | Triệu chứng | Sửa |
|---:|---|---|---|
| 1 | Quên "nối cả hai" | còn hai khoảng liền kề `[1,2],[3,3]` | kiểm tra `left_touch and right_touch` trước tiên |
| 2 | `bisect_left` thay cho `bisect_right` | khoảng bị lặp `[7,7],[7,7]` | `bisect_right(starts, value) - 1` |
| 3 | Thiếu `i >= 0` | số nhỏ bị "nuốt" vì `ends[-1]` | `i >= 0 and ...`, `i + 1 < len(...) and ...` |
| 4a | Cách 3 quên `seen` | khoảng chồng nhau `[1,3],[2,2]` | `if value in seen: return` |
| 4b | Cách 3 không `pop` khóa cũ | khoảng "ma" `[2,3],[3,3]` | `s = end_to_start.pop(value - 1)` |
| 4c | Cách 3 quên sort | kết quả không theo `start` | `sorted(start_to_end.items())` |
| 5 | Cách 1 quên ghi khoảng cuối sau vòng lặp | `[1,2,3,6,7]` → `[[1,3]]` | `result.append([s, e])` sau vòng `for` |
| 6 | `getIntervals` trả ra chính list nội bộ (vd `return self.intervals`) rồi người gọi sửa nó | trạng thái object bị hỏng ngầm | trả về list mới (như `[[s, e] for ...]`) |

## 13. Mẹo nhận dạng pattern, biến thể và bài liên quan

### Dấu hiệu nhận dạng

- Đề nói "**khoảng** / interval / range" và dữ liệu **thêm dần** (stream) → nghĩ ngay tới **giữ sẵn danh sách khoảng đã sắp xếp + bất biến không chồng, không liền kề**.
- Cần "khoảng **ngay trước** / **ngay sau** một điểm" → **floor / higher**: `bisect_right(...) - 1` / `bisect_right(...)` trong Python, `floorKey` / `higherKey` của `TreeMap` trong Java.
- Chỉ cần biết "có khoảng nào **kết thúc tại `x - 1`** / **bắt đầu tại `x + 1`**" → **dict hai đầu** (`O(1)`). Cùng mẹo này giải bài 128.
- Số nguyên ⇒ hai khoảng **liền kề** cũng phải gộp: so sánh với `+ 1`, đừng chỉ so sánh chồng nhau.

### Template trong đầu

1. Tìm `L` (start lớn nhất `<= value`) và `R` (khoảng ngay sau `L`).
2. Nằm trong `L`? → thôi.
3. Tính `left_touch`, `right_touch` → nối cả hai / nối trái / nối phải / đứng riêng.

### Biến thể

- **Union-Find** (DSU): coi mỗi số là một đỉnh, `addNum(v)` nối `v` với `v - 1` và `v + 1` nếu đã có; lưu `min`/`max` của mỗi nhóm. Cũng giải được, nhưng `getIntervals` vẫn phải sắp xếp các nhóm.
- **Thêm cả một khoảng** `addRange(left, right)` thay vì một số: lúc này một lần thêm có thể nuốt **nhiều** khoảng → dùng `bisect` tìm cả dải khoảng bị ảnh hưởng rồi thay bằng một khoảng (Bài 6 bên dưới).
- **Xóa số** khỏi stream: khoảng có thể bị **tách đôi** — bài 715 Range Module.

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 56. Merge Intervals | gộp khoảng tĩnh: sắp xếp theo start rồi quét (giống cách 1) |
| 57. Insert Interval | chèn một khoảng vào danh sách đã sắp xếp (giống Bài 6) |
| 228. Summary Ranges | phiên bản tĩnh của bài này: mảng đã sắp xếp → các khoảng (Bài 5) |
| 163. Missing Ranges | ngược lại: liệt kê các **khe hở** |
| 128. Longest Consecutive Sequence | dict hai đầu / set, tìm đoạn liên tiếp dài nhất |
| 715. Range Module | thêm / xóa / truy vấn khoảng — TreeMap / bisect |
| 2276. Count Integers in Intervals | thêm khoảng + đếm số được phủ |

## 14. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 15, đáp án ở mục 16 (cuối notebook). Các cell TODO bên dưới chỉ **định nghĩa** hàm/class; phần test đã được comment lại — làm xong thì bỏ comment để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

```python
import bisect
starts = [2, 5, 9]
print(bisect.bisect_right(starts, 5) - 1)
print(bisect.bisect_right(starts, 1) - 1)
print(bisect.bisect_left(starts, 5), bisect.bisect_right(starts, 5))
print(bisect.bisect_right(starts, 12) - 1)

obj = SummaryRanges()
for v in [4, 6, 5, 4, 10]:
    obj.addNum(v)
print(obj.getIntervals())
```

### Bài 2 — Tính tay bảng trạng thái của cách 2

Với chuỗi `addNum`: `5, 1, 3, 4, 2, 9, 8, 10, 4`, hãy lập bảng giống mục 6.1 với các cột: `value`, `i`, `L`, `R`, trường hợp (1–5), các khoảng sau bước đó. Chuỗi này đi qua **đủ cả 5 trường hợp** — trường hợp nào xuất hiện ở bước nào?

### Bài 3 — Điền TODO: `addNum` của cách 2

Hoàn thiện class dưới đây (không nhìn lại mục 6). Gợi ý cấu trúc đã có trong comment.

In [29]:
class SummaryRangesExercise:
    def __init__(self):
        self.starts = []
        self.ends = []

    def addNum(self, value):
        # TODO 1: i = chỉ số khoảng trái (start lớn nhất <= value), -1 nếu không có
        # TODO 2: nếu value nằm trong khoảng trái -> return
        # TODO 3: tính left_touch, right_touch (nhớ kiểm tra chỉ số hợp lệ trước)
        # TODO 4: xử lý theo thứ tự: nối cả hai -> nối trái -> nối phải -> đứng riêng
        pass

    def getIntervals(self):
        return [[s, e] for s, e in zip(self.starts, self.ends)]


# Tự làm xong rồi mới mở test:
# assert run_commands(SummaryRangesExercise, EX1_OPS, EX1_ARGS) == EX1_OUT
# run_basic_tests(SummaryRangesExercise)

### Bài 4 — Tự viết lại cách 3 (hai dict hai đầu)

Viết lại từ đầu class dùng `seen`, `start_to_end`, `end_to_start` với `addNum` `O(1)` trung bình. Thử viết sao cho **4 trường hợp gộp chung một đoạn code** (gợi ý: biến `s`, `e` mặc định bằng `value`).

In [30]:
class SummaryRangesTwoMapsExercise:
    def __init__(self):
        # TODO: tạo seen, start_to_end, end_to_start
        pass

    def addNum(self, value):
        # TODO
        raise NotImplementedError

    def getIntervals(self):
        # TODO: nhớ sắp xếp theo start
        raise NotImplementedError


# Gợi ý test:
# assert run_commands(SummaryRangesTwoMapsExercise, EX1_OPS, EX1_ARGS) == EX1_OUT
# run_basic_tests(SummaryRangesTwoMapsExercise)

### Bài 5 — Biến thể tĩnh: LeetCode 228. Summary Ranges

Cho mảng `nums` **đã sắp xếp tăng dần, không trùng**. Trả về list chuỗi tóm tắt các đoạn liên tiếp: đoạn nhiều số viết `"a->b"`, đoạn một số viết `"a"`.

- `[0, 1, 2, 4, 5, 7]` → `["0->2", "4->5", "7"]`
- `[0, 2, 3, 4, 6, 8, 9]` → `["0", "2->4", "6", "8->9"]`
- `[]` → `[]`

In [31]:
def summary_ranges_228_exercise(nums):
    # TODO: quét trái -> phải, gom đoạn liên tiếp (giống compress_sorted), rồi đổi sang chuỗi
    pass


# Gợi ý test:
# assert summary_ranges_228_exercise([0, 1, 2, 4, 5, 7]) == ["0->2", "4->5", "7"]
# assert summary_ranges_228_exercise([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]
# assert summary_ranges_228_exercise([]) == []
# assert summary_ranges_228_exercise([-3]) == ["-3"]

### Bài 6 — Mini challenge: thêm **cả một khoảng** `addRange(left, right)`

Mở rộng bài toán: ngoài `addNum(value)`, class có thêm `addRange(left, right)` thêm **mọi** số nguyên từ `left` tới `right` (`left <= right`). Giờ một lần thêm có thể nuốt **nhiều** khoảng cùng lúc, ví dụ đang có `[1,2], [4,5], [7,8], [12,12]` mà `addRange(3, 9)` → `[1, 9], [12, 12]`.

Yêu cầu: `addRange` dùng `bisect` để tìm **cả dải** khoảng bị ảnh hưởng (chồng hoặc liền kề với `[left, right]`) rồi thay dải đó bằng **một** khoảng. `addNum(value)` chỉ cần gọi `addRange(value, value)`.

In [32]:
class SummaryRangesWithRangeExercise:
    def __init__(self):
        self.starts = []
        self.ends = []

    def addRange(self, left, right):
        # TODO 1: lo = chỉ số khoảng ĐẦU TIÊN có end >= left - 1   (ends cũng tăng dần!)
        # TODO 2: hi = chỉ số ngay SAU khoảng CUỐI CÙNG có start <= right + 1
        # TODO 3: các khoảng lo..hi-1 bị nuốt; nới left/right theo chúng (nếu lo < hi)
        # TODO 4: thay dải [lo:hi] của starts/ends bằng một khoảng duy nhất
        pass

    def addNum(self, value):
        self.addRange(value, value)

    def getIntervals(self):
        return [[s, e] for s, e in zip(self.starts, self.ends)]


# Gợi ý test:
# obj = SummaryRangesWithRangeExercise()
# for l, r in [(1, 2), (4, 5), (7, 8), (12, 12)]:
#     obj.addRange(l, r)
# obj.addRange(3, 9)
# assert obj.getIntervals() == [[1, 9], [12, 12]]
# assert run_commands(SummaryRangesWithRangeExercise, EX1_OPS, EX1_ARGS) == EX1_OUT

## 15. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | `bisect_right` = số phần tử `<= x`; `bisect_left` = số phần tử `< x`. | Với `x = 5` có 2 số `<= 5` nhưng chỉ 1 số `< 5`. | Chuỗi `4, 6, 5`: số `5` chạm cả `[4,4]` và `[6,6]`. |
| 2 | Mỗi bước: tìm `L`, `R` → nằm trong? → chạm trái? chạm phải? | `i = bisect_right(starts, value) - 1`; `i = -1` khi `value` nhỏ hơn mọi start. | Bước 4 (`value = 4`): `L = [3,3]`, `R = [5,5]` → nối cả hai. |
| 3 | 3 bước: tìm → phân loại → sửa list. | `left_touch = i >= 0 and ends[i] + 1 == value`. | TH2: `ends[i] = ends[i+1]` rồi `pop(i+1)` ở **cả hai** list; TH5: `insert(i+1, value)` ở cả hai list. |
| 4 | Tra đầu mút bằng dict; số ở giữa khoảng thì nhờ `seen`. | `s = e = value`; nối trái thì `s = end_to_start.pop(value - 1)`. | Nối phải: `e = start_to_end.pop(value + 1)`; cuối cùng ghi cả `start_to_end[s] = e` và `end_to_start[e] = s`. |
| 5 | Đây chính là `compress_sorted` + đổi định dạng. | Giữ `s`, `e`; đóng khoảng khi `x != e + 1`. | `f"{s}->{e}" if s != e else str(s)`. |
| 6 | Khoảng `[s, e]` bị ảnh hưởng khi `e >= left - 1` **và** `s <= right + 1`. | `lo = bisect_left(ends, left - 1)`, `hi = bisect_right(starts, right + 1)`. | Nếu `lo < hi`: `left = min(left, starts[lo])`, `right = max(right, ends[hi-1])`; rồi `starts[lo:hi] = [left]`, `ends[lo:hi] = [right]`. |

## 16. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
1
-1
1 2
2
[[4, 6], [10, 10]]
```

- `bisect_right([2, 5, 9], 5) = 2` (hai số `<= 5`) → trừ 1 được `1`: khoảng trái của `5` là khoảng bắt đầu tại `5`.
- `bisect_right(..., 1) = 0` → `-1`: không có khoảng trái.
- `bisect_left(..., 5) = 1`, `bisect_right(..., 5) = 2`: khác nhau vì `5` có trong list.
- `bisect_right(..., 12) = 3` → `2`: khoảng cuối cùng.
- `4` → `[4,4]`; `6` → `[4,4],[6,6]`; `5` nối cả hai → `[4,6]`; `4` nằm trong; `10` đứng riêng.

### Đáp án Bài 2

| Bước | `value` | `i` | `L` | `R` | Trường hợp | Khoảng sau |
|---:|---:|---:|---|---|---|---|
| 1 | 5 | -1 | — | — | 5. đứng riêng | `[[5,5]]` |
| 2 | 1 | -1 | — | `[5,5]` | 5. đứng riêng | `[[1,1],[5,5]]` |
| 3 | 3 | 0 | `[1,1]` | `[5,5]` | 5. đứng riêng | `[[1,1],[3,3],[5,5]]` |
| 4 | 4 | 1 | `[3,3]` | `[5,5]` | 2. nối cả hai | `[[1,1],[3,5]]` |
| 5 | 2 | 0 | `[1,1]` | `[3,5]` | 2. nối cả hai | `[[1,5]]` |
| 6 | 9 | 0 | `[1,5]` | — | 5. đứng riêng | `[[1,5],[9,9]]` |
| 7 | 8 | 0 | `[1,5]` | `[9,9]` | 4. nối phải | `[[1,5],[8,9]]` |
| 8 | 10 | 1 | `[8,9]` | — | 3. nối trái | `[[1,5],[8,10]]` |
| 9 | 4 | 0 | `[1,5]` | `[8,10]` | 1. nằm trong | `[[1,5],[8,10]]` |

Bước 2 là ví dụ `i = -1`: số `1` nhỏ hơn mọi start. Bước 4 và 5 là hai cú "hàn" liên tiếp.

Cell kế tiếp kiểm chứng Bài 1, Bài 2 bằng code thật, rồi đưa **đáp án Bài 3–6** kèm assert.

In [33]:
# ---------- Kiểm chứng Bài 1 ----------
starts_ex = [2, 5, 9]
assert bisect.bisect_right(starts_ex, 5) - 1 == 1
assert bisect.bisect_right(starts_ex, 1) - 1 == -1
assert (bisect.bisect_left(starts_ex, 5), bisect.bisect_right(starts_ex, 5)) == (1, 2)
assert bisect.bisect_right(starts_ex, 12) - 1 == 2
assert check_tokens(SummaryRanges, [4, 6, 5, 4, 10], [[4, 6], [10, 10]])

# ---------- Kiểm chứng Bài 2 ----------
assert trace_bisect([5, 1, 3, 4, 2, 9, 8, 10, 4]) == [[1, 5], [8, 10]]

bước value   i  L        R        trường hợp       khoảng sau
   1     5  -1  -        -        5. đứng riêng    [[5, 5]]
   2     1  -1  -        [5,5]    5. đứng riêng    [[1, 1], [5, 5]]
   3     3   0  [1,1]    [5,5]    5. đứng riêng    [[1, 1], [3, 3], [5, 5]]
   4     4   1  [3,3]    [5,5]    2. nối cả hai    [[1, 1], [3, 5]]
   5     2   0  [1,1]    [3,5]    2. nối cả hai    [[1, 5]]
   6     9   0  [1,5]    -        5. đứng riêng    [[1, 5], [9, 9]]
   7     8   0  [1,5]    [9,9]    4. nối phải      [[1, 5], [8, 9]]
   8    10   1  [8,9]    -        3. nối trái      [[1, 5], [8, 10]]
   9     4   0  [1,5]    [8,10]   1. nằm trong     [[1, 5], [8, 10]]


In [34]:
# ---------- Bài 3 ----------
class SummaryRangesAnswer3:
    def __init__(self):
        self.starts = []
        self.ends = []

    def addNum(self, value):
        starts, ends = self.starts, self.ends
        i = bisect.bisect_right(starts, value) - 1
        if i >= 0 and value <= ends[i]:
            return
        left_touch = i >= 0 and ends[i] + 1 == value
        right_touch = i + 1 < len(starts) and starts[i + 1] == value + 1
        if left_touch and right_touch:
            ends[i] = ends[i + 1]
            starts.pop(i + 1)
            ends.pop(i + 1)
        elif left_touch:
            ends[i] = value
        elif right_touch:
            starts[i + 1] = value
        else:
            starts.insert(i + 1, value)
            ends.insert(i + 1, value)

    def getIntervals(self):
        return [[s, e] for s, e in zip(self.starts, self.ends)]


# ---------- Bài 4 ----------
class SummaryRangesTwoMapsAnswer4:
    def __init__(self):
        self.seen = set()
        self.start_to_end = {}
        self.end_to_start = {}

    def addNum(self, value):
        if value in self.seen:
            return
        self.seen.add(value)
        s = e = value
        if value - 1 in self.end_to_start:
            s = self.end_to_start.pop(value - 1)
        if value + 1 in self.start_to_end:
            e = self.start_to_end.pop(value + 1)
        self.start_to_end[s] = e          # ghi đè luôn khóa s cũ (nếu nối trái)
        self.end_to_start[e] = s          # ghi đè luôn khóa e cũ (nếu nối phải)

    def getIntervals(self):
        return [[s, e] for s, e in sorted(self.start_to_end.items())]


# ---------- Bài 5 ----------
def summary_ranges_228_answer(nums):
    result = []
    for s, e in compress_sorted(nums):
        result.append(f"{s}->{e}" if s != e else str(s))
    return result


# ---------- Bài 6 ----------
class SummaryRangesWithRangeAnswer:
    def __init__(self):
        self.starts = []
        self.ends = []

    def addRange(self, left, right):
        lo = bisect.bisect_left(self.ends, left - 1)        # khoảng đầu tiên có end >= left - 1
        hi = bisect.bisect_right(self.starts, right + 1)    # sau khoảng cuối có start <= right + 1
        if lo < hi:                                         # có khoảng bị nuốt
            left = min(left, self.starts[lo])
            right = max(right, self.ends[hi - 1])
        self.starts[lo:hi] = [left]                         # thay cả dải bằng 1 khoảng
        self.ends[lo:hi] = [right]

    def addNum(self, value):
        self.addRange(value, value)

    def getIntervals(self):
        return [[s, e] for s, e in zip(self.starts, self.ends)]


# ---------- Test ----------
for cls in [SummaryRangesAnswer3, SummaryRangesTwoMapsAnswer4, SummaryRangesWithRangeAnswer]:
    assert run_commands(cls, EX1_OPS, EX1_ARGS) == EX1_OUT
    run_basic_tests(cls)

assert summary_ranges_228_answer([0, 1, 2, 4, 5, 7]) == ["0->2", "4->5", "7"]
assert summary_ranges_228_answer([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]
assert summary_ranges_228_answer([]) == []
assert summary_ranges_228_answer([-3]) == ["-3"]

obj = SummaryRangesWithRangeAnswer()
for l, r in [(1, 2), (4, 5), (7, 8), (12, 12)]:
    obj.addRange(l, r)
obj.addRange(3, 9)
assert obj.getIntervals() == [[1, 9], [12, 12]]

# Bài 6: đối chiếu với set "vét cạn" trên các addRange ngẫu nhiên (seed cố định)
rng = random.Random(6)
for _ in range(300):
    obj, seen = SummaryRangesWithRangeAnswer(), set()
    for _ in range(rng.randint(1, 15)):
        l = rng.randint(0, 40)
        r = l + rng.randint(0, 5)
        obj.addRange(l, r)
        seen.update(range(l, r + 1))
        check_invariant(obj.starts, obj.ends, seen)
    assert obj.getIntervals() == compress_sorted(sorted(seen))
print("Đáp án tham khảo Bài 1–6 chạy PASS toàn bộ test!")

Đáp án tham khảo Bài 1–6 chạy PASS toàn bộ test!


## 17. Template cuối cùng nên nhớ

**Cách chính — list khoảng đã sắp xếp + `bisect`**: `addNum` `O(log k)` tìm + `O(k)` dịch tệ nhất, `getIntervals` `O(k)`, bộ nhớ `O(k)`:

```python
import bisect

class SummaryRanges:
    def __init__(self):
        self.starts, self.ends = [], []

    def addNum(self, value: int) -> None:
        st, en = self.starts, self.ends
        i = bisect.bisect_right(st, value) - 1            # khoảng trái L
        if i >= 0 and value <= en[i]:                      # 1. nằm trong
            return
        left = i >= 0 and en[i] + 1 == value
        right = i + 1 < len(st) and st[i + 1] == value + 1
        if left and right:                                 # 2. nối cả hai
            en[i] = en[i + 1]; st.pop(i + 1); en.pop(i + 1)
        elif left:                                         # 3. nối trái
            en[i] = value
        elif right:                                        # 4. nối phải
            st[i + 1] = value
        else:                                              # 5. đứng riêng
            st.insert(i + 1, value); en.insert(i + 1, value)

    def getIntervals(self) -> List[List[int]]:
        return [[s, e] for s, e in zip(self.starts, self.ends)]
```

**Câu thần chú:** *"Tìm khoảng trái bằng `bisect_right - 1`; nằm trong thì thôi; chạm hai bên thì hàn, chạm một bên thì kéo dài, không chạm thì chèn."*

**Bất biến:** khoảng sắp xếp theo start; giữa hai khoảng kề nhau luôn có **khe hở** (`end + 1 < start kế tiếp`).

**Cách 3 (addNum `O(1)`):** *"`seen` chặn số cũ; `end_to_start[value-1]` nối trái, `start_to_end[value+1]` nối phải; `pop` khóa cũ; `sorted` khi hỏi."*

**Follow-up:** *"Chỉ lưu khoảng trong map có thứ tự (TreeMap / `SortedDict`) → `O(log k)` mỗi lần thêm, `O(k)` bộ nhớ."*

## 18. Checklist tự kiểm tra trước khi submit

- Tên class và method đúng y như đề: `SummaryRanges`, `addNum(self, value)`, `getIntervals(self)`?
- Mọi trạng thái được tạo **trong `__init__`** (không khai báo list ở mức class, kẻo mọi object dùng chung)?
- Dùng `bisect_right(starts, value) - 1` (không phải `bisect_left`) để tìm khoảng trái?
- Có `i >= 0 and ...` trước khi đọc `ends[i]`, và `i + 1 < len(starts) and ...` trước khi đọc `starts[i + 1]` (bẫy `ends[-1]`)?
- Kiểm tra "nằm trong" **đầu tiên** (xử lý số trùng), rồi "nối cả hai" **trước** nối trái / nối phải?
- Khi nối cả hai: sửa `ends[i]` rồi `pop(i + 1)` ở **cả hai** list; khi đứng riêng: `insert(i + 1, ...)` ở **cả hai** list?
- `getIntervals` trả về list các list `[start, end]` **sắp xếp theo start** (cách dict phải `sorted`)?
- Đã test: Example 1, một số, số trùng, số nằm giữa khoảng, `0` và `10^4`, tăng dần / giảm dần, số nhỏ hơn mọi khoảng, cú "hàn" hai khoảng?
- Nói được độ phức tạp **từng thao tác** và trả lời được follow-up (chỉ lưu khoảng, map có thứ tự, `O(k)` bộ nhớ)?